# MedQA confidence signals — combine models (CPU only)

Add each model's run output (or its `share_bundle_<model>.zip`) as notebook input, then Save & Run All.
Everything is recomputed from the per-model `per_question.csv` files; Holm correction is applied across all
pairs × non-flagged models within each setting (CoT = primary family). Output: `/kaggle/working/combined/share_bundle_all.zip`.


In [ ]:
import os
os.makedirs('medqa_src/medqa_conf', exist_ok=True)

In [ ]:
%%writefile medqa_src/config.py
"""Pre-registered configuration. Do not change values without amending PREREGISTRATION.md.

Everything that determines model outputs or analysis results lives here. The config hash
(used for resume and logged in run_log.json) covers every field of `Config`.
"""
from __future__ import annotations

import dataclasses
import hashlib
import json

DATASET_ID = "GBaker/MedQA-USMLE-4-options"
TEST_ROWS = 1273
TRAIN_ROWS = 10178
LETTERS = ("A", "B", "C", "D")

DEFAULT_MODELS = (
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
)


@dataclasses.dataclass(frozen=True)
class Config:
    model_id: str = DEFAULT_MODELS[0]
    pilot: bool = False
    dataset_id: str = DATASET_ID
    # generation
    base_seed: int = 12345
    greedy_max_new_tokens: int = 1024
    n_samples: int = 5
    sample_temperature: float = 0.7
    sample_top_p: float = 1.0
    sample_max_new_tokens: int = 1024
    confidence_max_new_tokens: int = 6
    topk_logprobs: int = 20
    dtype: str = "float16"
    max_model_len: int = 4096
    tensor_parallel_size: int = 2
    # pilot
    pilot_n: int = 20
    pilot_split: str = "train"
    max_projected_hours: float = 11.0
    # analysis
    n_bins: int = 15
    coverage: float = 0.8
    n_boot: int = 2000
    boot_seed: int = 2026
    cv_folds: int = 5
    cv_seed: int = 2026
    alpha: float = 0.05
    # validity flags
    flag_min_accuracy: float = 0.40
    flag_max_parse_fail: float = 0.10
    flag_min_median_letter_mass: float = 0.5
    flag_max_truncation: float = 0.10
    # engineering
    checkpoint_every: int = 50
    determinism_n: int = 10

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)

    def config_hash(self) -> str:
        blob = json.dumps(self.to_dict(), sort_keys=True).encode()
        return hashlib.sha256(blob).hexdigest()[:12]

    @property
    def model_slug(self) -> str:
        slug = self.model_id.split("/")[-1].replace(".", "_").replace("-", "_").lower()
        return slug + ("_pilot" if self.pilot else "")


# Analysis-only settings (used by report.py when recomputing from a CSV).
ANALYSIS_KEYS = ("n_bins", "coverage", "n_boot", "boot_seed", "cv_folds", "cv_seed", "alpha",
                 "flag_min_accuracy", "flag_max_parse_fail", "flag_min_median_letter_mass",
                 "flag_max_truncation")


In [ ]:
%%writefile medqa_src/medqa_conf/__init__.py
"""MedQA confidence-signal study."""
__version__ = "1.0.0"


In [ ]:
%%writefile medqa_src/medqa_conf/data.py
"""Dataset loading, fail-loud validation and light cleaning."""
from __future__ import annotations

import collections
import dataclasses
import hashlib

import numpy as np

LETTERS = ("A", "B", "C", "D")


class DataValidationError(RuntimeError):
    pass


@dataclasses.dataclass
class Question:
    qid: int
    question: str
    options: dict  # letter -> cleaned text
    gold: str
    answer_text: str
    meta_info: str
    qhash: str  # sha1 of the raw question text


def clean_option(text: str) -> str:
    """Strip whitespace and a stray (unbalanced) trailing double quote."""
    out = text.strip()
    while out.endswith('"') and out.count('"') % 2 == 1:
        out = out[:-1].strip()
    return out


def validate_rows(rows: list[dict], expected_n: int | None, split: str) -> None:
    if expected_n is not None and len(rows) != expected_n:
        raise DataValidationError(f"{split}: expected exactly {expected_n} rows, got {len(rows)}")
    for i, r in enumerate(rows):
        opts = r.get("options")
        if not isinstance(opts, dict) or sorted(opts.keys()) != list(LETTERS):
            raise DataValidationError(f"{split} row {i}: options keys {None if not isinstance(opts, dict) else sorted(opts)} != A-D")
        if any(not isinstance(v, str) or not v.strip() for v in opts.values()):
            raise DataValidationError(f"{split} row {i}: empty option text")
        if r.get("answer_idx") not in LETTERS:
            raise DataValidationError(f"{split} row {i}: gold letter {r.get('answer_idx')!r} not in A-D")
        if not isinstance(r.get("question"), str) or not r["question"].strip():
            raise DataValidationError(f"{split} row {i}: empty question")


def build_questions(rows: list[dict], indices=None) -> tuple[list[Question], dict]:
    """Clean rows into Questions. `indices` selects rows (qid = row index)."""
    if indices is None:
        indices = range(len(rows))
    qs, n_cleaned, answer_text_mismatch = [], 0, 0
    for i in indices:
        r = rows[i]
        opts = {}
        for L in LETTERS:
            raw = r["options"][L]
            c = clean_option(raw)
            n_cleaned += int(c != raw)
            opts[L] = c
        gold = r["answer_idx"]
        ans = clean_option(r.get("answer", "") or "")
        answer_text_mismatch += int(ans != opts[gold])
        qs.append(Question(qid=int(i), question=r["question"].strip(), options=opts, gold=gold,
                           answer_text=ans, meta_info=str(r.get("meta_info", "")),
                           qhash=hashlib.sha1(r["question"].encode("utf-8")).hexdigest()))
    stats = {
        "n_questions": len(qs),
        "n_options_cleaned": n_cleaned,
        "n_answer_text_mismatch": answer_text_mismatch,
        "gold_distribution": dict(sorted(collections.Counter(q.gold for q in qs).items())),
        "meta_info_distribution": dict(sorted(collections.Counter(q.meta_info for q in qs).items())),
    }
    return qs, stats


def pilot_indices(n_train: int, n: int, seed: int) -> list[int]:
    rng = np.random.default_rng(seed)
    return sorted(int(i) for i in rng.choice(n_train, size=n, replace=False))


def load_hf_splits(dataset_id: str, cache_dir: str | None = None, revision: str | None = None):
    """Return (train_rows, test_rows, dataset_revision). Needs `datasets` + network."""
    import datasets
    from huggingface_hub import HfApi

    if revision is None:
        try:
            revision = HfApi().dataset_info(dataset_id).sha
        except Exception:  # offline / API hiccup: still load, but record it
            revision = None
    ds = datasets.load_dataset(dataset_id, cache_dir=cache_dir, revision=revision)
    return [dict(r) for r in ds["train"]], [dict(r) for r in ds["test"]], revision


def load_questions(cfg, train_rows: list[dict], test_rows: list[dict],
                   expected_test=1273, expected_train=10178) -> tuple[list[Question], dict]:
    """Validate and select the questions for this run (test for the main run, train for the pilot)."""
    validate_rows(test_rows, expected_test, "test")  # always fail loudly on the test schema
    test_qs, test_stats = build_questions(test_rows)
    info = {"test": test_stats, "n_train_rows": len(train_rows)}
    if expected_train is not None and len(train_rows) != expected_train:
        info["warning_train_rows"] = f"expected {expected_train} train rows, got {len(train_rows)}"
    if cfg.pilot:
        idx = pilot_indices(len(train_rows), cfg.pilot_n, cfg.base_seed)
        sub = [train_rows[i] for i in idx]
        validate_rows(sub, None, "train(pilot)")
        qs, st = build_questions(train_rows, idx)
        info["used_split"] = "train"
        info["pilot_train_indices"] = idx
        info["used"] = st
        return qs, info
    info["used_split"] = "test"
    info["used"] = test_stats
    return test_qs, info


def synthetic_rows(n: int, seed: int = 0) -> list[dict]:
    """Fake MedQA-shaped rows for CPU tests (some options carry the trailing '\\n\"' defect)."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        opts = {L: f"Option {L} for case {i}: finding {rng.integers(1000)}" for L in LETTERS}
        if i % 7 == 0:
            opts["C"] = opts["C"] + '\n"'
        gold = LETTERS[int(rng.integers(4))]
        rows.append({
            "question": f"A {20 + i % 60}-year-old patient presents with symptom #{i}. Which is the best next step?",
            "options": opts, "answer_idx": gold, "answer": clean_option(opts[gold]),
            "meta_info": "step1" if i % 2 == 0 else "step2&3",
        })
    return rows


In [ ]:
%%writefile medqa_src/medqa_conf/prompts.py
"""Pre-registered prompts (verbatim from PREREGISTRATION.md section 4)."""
from __future__ import annotations

QUESTION_BLOCK = (
    "The following is a multiple-choice question from the US Medical Licensing Examination (USMLE). "
    "Choose the single best answer.\n"
    "\n"
    "Question: {question}\n"
    "A. {A}\n"
    "B. {B}\n"
    "C. {C}\n"
    "D. {D}"
)

COT_SUFFIX = (
    "\n"
    "\n"
    "Think through the problem step by step. Then finish with exactly these two lines:\n"
    "Answer: <A, B, C or D>\n"
    "Confidence: <a number from 0 to 100: the probability that your answer is correct>"
)

DIRECT_SUFFIX = (
    "\n"
    "\n"
    "Reply with exactly these two lines and nothing else:\n"
    "Answer: <A, B, C or D>\n"
    "Confidence: <a number from 0 to 100: the probability that your answer is correct>"
)

ANSWER_ANCHOR = "Answer:"


def question_block(q) -> str:
    return QUESTION_BLOCK.format(question=q.question, **q.options)


def cot_message(q) -> str:
    return question_block(q) + COT_SUFFIX


def direct_message(q) -> str:
    return question_block(q) + DIRECT_SUFFIX


def cot_letter_prefix(greedy_text: str) -> tuple[str, bool]:
    """G up to and including its last 'Answer:'; else G.rstrip() + '\\nAnswer:' (forced)."""
    pos = greedy_text.rfind(ANSWER_ANCHOR)
    if pos >= 0:
        return greedy_text[: pos + len(ANSWER_ANCHOR)], False
    return greedy_text.rstrip() + "\n" + ANSWER_ANCHOR, True


def cot_recovery_prefill(cot_prefix: str, letter: str) -> str:
    return cot_prefix + f" {letter}\nConfidence:"


DIRECT_LETTER_PREFILL = "Answer:"


def direct_conf_prefill(letter: str) -> str:
    return f"Answer: {letter}\nConfidence:"


In [ ]:
%%writefile medqa_src/medqa_conf/parsing.py
"""Answer / confidence parsing (PREREGISTRATION.md sections 5-6)."""
from __future__ import annotations

import math
import re

# "Answer", optional "**" before the colon, ":", optional "**", optional "(" or "[", one capital A-D
# that is not followed by another letter. Last match wins.
ANSWER_LINE_RE = re.compile(r"(?i:answer)(?:\s*\*\*)?\s*:\s*(?:\*\*\s*)?[\(\[]?\s*([A-D])(?![A-Za-z])")
ANSWER_IS_RE = re.compile(
    r"(?i:answer\s+is)\s*(?:\*\*\s*)?(?:(?i:option)\s*)?[:\(\[]?\s*(?:\*\*\s*)?([A-D])(?![A-Za-z])")

_NUM = r"(\d+(?:\.\d+)?|\.\d+)"
CONF_RE = re.compile(r"(?i:confidence)(?:\s*\*\*)?\s*:\s*(?:\*\*\s*)?" + _NUM + r"\s*%?")
LEADING_NUM_RE = re.compile(r"^\s*(?:\*\*\s*)?" + _NUM)


def parse_answer(text: str | None) -> tuple[str | None, str]:
    """Return (letter or None, method) with method in {'answer_line', 'answer_is', 'none'}."""
    if not text:
        return None, "none"
    m = list(ANSWER_LINE_RE.finditer(text))
    if m:
        return m[-1].group(1), "answer_line"
    m = list(ANSWER_IS_RE.finditer(text))
    if m:
        return m[-1].group(1), "answer_is"
    return None, "none"


def _to_conf(num: str) -> tuple[float, bool]:
    v = float(num)
    if v <= 1 and "." in num:
        return min(max(v, 0.0), 1.0), True
    return min(max(v / 100.0, 0.0), 1.0), False


def parse_confidence(text: str | None) -> tuple[float, bool]:
    """Last 'Confidence:' number -> (conf in [0,1] or NaN, fraction_flag)."""
    if not text:
        return math.nan, False
    m = list(CONF_RE.finditer(text))
    if not m:
        return math.nan, False
    return _to_conf(m[-1].group(1))


def parse_confidence_continuation(text: str | None) -> tuple[float, bool]:
    """Number at the start of a short generation that follows a 'Confidence:' prefill."""
    if not text:
        return math.nan, False
    m = LEADING_NUM_RE.match(text)
    if not m:
        return math.nan, False
    return _to_conf(m.group(1))


In [ ]:
%%writefile medqa_src/medqa_conf/backends.py
"""Model backends behind one interface: vLLM (preferred), transformers (fallback), Mock (CPU tests).

Interface
---------
render(user_msg) -> str                       chat-template prompt (no system message, generation prompt added)
generate(list[GenRequest]) -> list[list[GenOutput]]
next_token_logprobs(list[str]) -> list[dict[token_id, logprob]]   restricted to letter-variant ids that are
                                              available (top-k for vLLM, full softmax for HF). Raises on NaN/inf.
letter_variants: dict[letter, list[token_id]] ; letter_variant_strings: dict[token_id, str]
info() -> dict
"""
from __future__ import annotations

import dataclasses
import hashlib
import math
import re

LETTERS = ("A", "B", "C", "D")


class NonFiniteLogprobError(RuntimeError):
    pass


@dataclasses.dataclass
class GenRequest:
    prompt: str
    max_new_tokens: int
    temperature: float = 0.0
    top_p: float = 1.0
    n: int = 1
    seed: int | None = None


@dataclasses.dataclass
class GenOutput:
    text: str
    n_tokens: int
    finish_reason: str  # "stop" | "length"


def check_finite(values, where: str) -> None:
    for v in values:
        if not math.isfinite(v):
            raise NonFiniteLogprobError(f"non-finite logprob ({v}) in {where}")


def find_letter_variants(tokenizer) -> tuple[dict, dict]:
    """Single-token ids for 'L' and ' L' (byte-level 'ĠL' or sentencepiece '▁L')."""
    vocab = tokenizer.get_vocab()
    variants, strings = {}, {}
    for L in LETTERS:
        ids = []
        for cand in (L, "Ġ" + L, "▁" + L):
            tid = vocab.get(cand)
            if tid is None or tid in ids:
                continue
            dec = tokenizer.decode([tid])
            if dec.strip() == L:
                ids.append(tid)
                strings[tid] = dec
        if not ids:
            raise RuntimeError(f"no single-token variant found for letter {L}")
        variants[L] = ids
    return variants, strings


def render_chat(tokenizer, user_msg: str) -> str:
    msgs = [{"role": "user", "content": user_msg}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                             enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def eos_token_ids(model_path: str, tokenizer) -> list[int]:
    ids = []
    try:
        from transformers import GenerationConfig
        e = GenerationConfig.from_pretrained(model_path).eos_token_id
        ids += list(e) if isinstance(e, (list, tuple)) else ([e] if e is not None else [])
    except Exception:
        pass
    if tokenizer.eos_token_id is not None:
        ids.append(tokenizer.eos_token_id)
    return sorted(set(int(i) for i in ids))


def download_model(model_id: str, cache_dir: str, revision: str | None = None, token: str | None = None):
    """Resolve the exact commit hash and download weights (safetensors shards only). -> (path, sha)."""
    from huggingface_hub import HfApi, snapshot_download
    sha = HfApi(token=token).model_info(model_id, revision=revision).sha
    path = snapshot_download(model_id, revision=sha, cache_dir=cache_dir, token=token,
                             allow_patterns=["*.json", "model*.safetensors", "*.model", "tokenizer*",
                                             "*.txt", "*.jinja", "*.tiktoken"])
    return path, sha


class Backend:
    name = "base"
    topk: int | None = None
    letter_variants: dict
    letter_variant_strings: dict

    def render(self, user_msg: str) -> str:
        return render_chat(self.tokenizer, user_msg)

    def all_letter_ids(self) -> list[int]:
        return [i for L in LETTERS for i in self.letter_variants[L]]

    def info(self) -> dict:
        return {"backend": self.name, "topk": self.topk,
                "letter_variants": {L: [[i, self.letter_variant_strings[i]] for i in ids]
                                    for L, ids in self.letter_variants.items()}}


# --------------------------------------------------------------------------------------------- vLLM
class VLLMBackend(Backend):
    name = "vllm"

    def __init__(self, model_path: str, cfg, gpu_memory_utilization: float = 0.90):
        from transformers import AutoTokenizer
        from vllm import LLM

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.eos_ids = eos_token_ids(model_path, self.tokenizer)
        self.letter_variants, self.letter_variant_strings = find_letter_variants(self.tokenizer)
        self.topk = cfg.topk_logprobs
        kw = dict(model=model_path, tokenizer=model_path, tensor_parallel_size=cfg.tensor_parallel_size,
                  dtype=cfg.dtype, max_model_len=cfg.max_model_len, seed=cfg.base_seed,
                  gpu_memory_utilization=gpu_memory_utilization, disable_custom_all_reduce=True,
                  max_logprobs=max(20, cfg.topk_logprobs), generation_config="vllm")
        try:
            self.llm = LLM(**kw)
        except TypeError as e:  # older vLLM without `generation_config`
            if "generation_config" not in str(e):
                raise
            kw.pop("generation_config")
            self.llm = LLM(**kw)

    def _ids(self, prompt: str) -> dict:
        return {"prompt_token_ids": self.tokenizer.encode(prompt, add_special_tokens=False)}

    def generate(self, reqs):
        from vllm import SamplingParams
        sps = [SamplingParams(n=r.n, temperature=r.temperature, top_p=r.top_p, max_tokens=r.max_new_tokens,
                              seed=r.seed, stop_token_ids=self.eos_ids, repetition_penalty=1.0,
                              skip_special_tokens=True) for r in reqs]
        outs = self.llm.generate([self._ids(r.prompt) for r in reqs], sps, use_tqdm=False)
        res = []
        for o in outs:
            comps = sorted(o.outputs, key=lambda c: c.index)
            res.append([GenOutput(text=c.text, n_tokens=len(c.token_ids),
                                  finish_reason="length" if c.finish_reason == "length" else "stop")
                        for c in comps])
        return res

    def next_token_logprobs(self, prompts):
        from vllm import SamplingParams
        sp = SamplingParams(max_tokens=1, temperature=0.0, logprobs=self.topk)
        outs = self.llm.generate([self._ids(p) for p in prompts], sp, use_tqdm=False)
        wanted = set(self.all_letter_ids())
        res = []
        for o in outs:
            lp = o.outputs[0].logprobs[0]  # dict token_id -> Logprob (top-k, plus sampled token)
            if not hasattr(lp, "items"):
                raise RuntimeError(f"unexpected vLLM logprobs container {type(lp)}; unset VLLM_FLAT_LOGPROBS")
            vals = {int(t): float(getattr(v, "logprob", v)) for t, v in lp.items()}
            check_finite(vals.values(), "vLLM top-k logprobs")
            res.append({t: v for t, v in vals.items() if t in wanted})
        return res

    def info(self):
        import vllm
        d = super().info()
        d.update(vllm_version=vllm.__version__, eos_ids=self.eos_ids)
        return d


# --------------------------------------------------------------------------------------- transformers
class HFBackend(Backend):
    """transformers fallback with its own KV-cache decoding loop, so that sampled requests from several
    questions can share a batch while each question keeps its own torch.Generator(seed=BASE_SEED+qid)."""
    name = "hf"

    def __init__(self, model_path: str, cfg, batch_size: int = 16, sample_batch_questions: int = 4,
                 device_map="auto", dtype=None):
        import torch
        import transformers
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.eos_ids = eos_token_ids(model_path, self.tokenizer)
        self.pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else self.eos_ids[0]
        self.letter_variants, self.letter_variant_strings = find_letter_variants(self.tokenizer)
        self.topk = None  # full distribution
        dtype = dtype or getattr(torch, cfg.dtype)
        major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
        dkey = "dtype" if (major, minor) >= (4, 56) else "torch_dtype"  # kwarg renamed in 4.56
        self.model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device_map, **{dkey: dtype})
        if next(self.model.parameters()).dtype != dtype:
            raise RuntimeError(f"model loaded in {next(self.model.parameters()).dtype}, expected {dtype}")
        self.model.eval()
        self.batch_size = batch_size
        self.sample_batch_questions = sample_batch_questions

    def _in_device(self):
        return self.model.get_input_embeddings().weight.device

    def _batch(self, prompts):
        enc = [self.tokenizer.encode(p, add_special_tokens=False) for p in prompts]
        L = max(len(e) for e in enc)
        ids = [[self.pad_id] * (L - len(e)) + e for e in enc]
        att = [[0] * (L - len(e)) + [1] * len(e) for e in enc]
        t, dev = self.torch, self._in_device()
        ids, att = t.tensor(ids, device=dev), t.tensor(att, device=dev)
        pos = (att.cumsum(-1) - 1).clamp(min=0)  # left padding -> explicit position ids
        return ids, att, pos

    def _forward_last(self, ids, att, pos, past=None, use_cache=False):
        """Logits of the last position only (avoids a B x L x vocab logits tensor)."""
        out = self.model.base_model(input_ids=ids, attention_mask=att, position_ids=pos,
                                    past_key_values=past, use_cache=use_cache)
        logits = self.model.get_output_embeddings()(out.last_hidden_state[:, -1, :]).float()
        if not bool(self.torch.isfinite(logits).all()):
            raise NonFiniteLogprobError("non-finite logits in HF forward pass")
        return logits, (out.past_key_values if use_cache else None)

    def _decode(self, prompts, max_new, temperature, top_p, seeds):
        """Greedy (temperature 0) or ancestral sampling. seeds[i] identifies row i's generator; rows with
        the same seed share one generator (the n samples of one question). Every row draws at every step,
        so a question's samples do not depend on which other questions share the batch."""
        t = self.torch
        ids, att, pos = self._batch(prompts)
        B = ids.shape[0]
        eos = set(self.eos_ids)
        toks = [[] for _ in range(B)]
        finished, stopped = [False] * B, [False] * B
        gens = None
        with t.no_grad():
            logits, past = self._forward_last(ids, att, pos, use_cache=True)
            last_pos = pos[:, -1:]
            for step in range(max_new):
                if temperature == 0:
                    nxt = logits.argmax(-1).tolist()
                else:
                    probs = t.softmax(logits / temperature, dim=-1)
                    if top_p < 1.0:
                        sp, si = probs.sort(-1, descending=True)
                        sp[(sp.cumsum(-1) - sp) > top_p] = 0
                        probs = t.zeros_like(probs).scatter(-1, si, sp)
                        probs = probs / probs.sum(-1, keepdim=True)
                    if gens is None:
                        made = {}
                        for sd in seeds:
                            if sd not in made:
                                made[sd] = t.Generator(device=probs.device).manual_seed(int(sd))
                        gens = [made[sd] for sd in seeds]
                    nxt = [int(t.multinomial(probs[i], 1, generator=gens[i])) for i in range(B)]
                for i in range(B):
                    if finished[i]:
                        continue
                    if nxt[i] in eos:
                        finished[i] = stopped[i] = True
                    else:
                        toks[i].append(nxt[i])
                        finished[i] = len(toks[i]) >= max_new
                if all(finished):
                    break
                step_ids = t.tensor([[self.pad_id if stopped[i] else nxt[i]] for i in range(B)], device=ids.device)
                att = t.cat([att, t.ones((B, 1), dtype=att.dtype, device=att.device)], dim=-1)
                last_pos = last_pos + 1
                logits, past = self._forward_last(step_ids, att, last_pos, past, use_cache=True)
        return [GenOutput(self.tokenizer.decode(toks[i], skip_special_tokens=True), len(toks[i]),
                          "stop" if stopped[i] else "length") for i in range(B)]

    def generate(self, reqs):
        res = [None] * len(reqs)
        groups = {}
        for i, r in enumerate(reqs):
            key = (r.max_new_tokens, r.temperature, r.top_p, r.temperature == 0 and r.n == 1)
            groups.setdefault(key, []).append(i)
        for (max_new, temp, top_p, greedy), idxs in groups.items():
            step = self.batch_size if greedy else self.sample_batch_questions
            for b in range(0, len(idxs), step):
                chunk = idxs[b:b + step]
                rows, seeds, owner = [], [], []
                for i in chunk:
                    r = reqs[i]
                    for j in range(r.n):
                        rows.append(r.prompt)
                        seeds.append(r.seed if r.seed is not None else 0)
                        owner.append(i)
                outs = self._decode(rows, max_new, temp, top_p, seeds)
                for i in chunk:
                    res[i] = [o for o, w in zip(outs, owner) if w == i]
        return res

    def next_token_logprobs(self, prompts):
        t = self.torch
        wanted = self.all_letter_ids()
        res = []
        for b in range(0, len(prompts), self.batch_size):
            ids, att, pos = self._batch(prompts[b:b + self.batch_size])
            with t.no_grad():
                logits, _ = self._forward_last(ids, att, pos)
            lp = t.log_softmax(logits, dim=-1)
            sel = lp[:, wanted].cpu().tolist()
            for row in sel:
                check_finite(row, "HF letter logprobs")
                res.append(dict(zip(wanted, row)))
        return res

    def info(self):
        import transformers
        d = super().info()
        d.update(transformers_version=transformers.__version__, eos_ids=self.eos_ids)
        return d


# --------------------------------------------------------------------------------------------- Mock
def _u(*parts) -> float:
    """Deterministic uniform(0,1) from a hash of the parts."""
    h = hashlib.sha256("|".join(map(str, parts)).encode()).digest()
    return int.from_bytes(h[:8], "big") / 2 ** 64


class _MockTokenizer:
    def __init__(self):
        self.vocab = {L: i + 1 for i, L in enumerate(LETTERS)}
        self.vocab.update({"Ġ" + L: i + 11 for i, L in enumerate(LETTERS)})
        self.eos_token_id = 0

    def get_vocab(self):
        return dict(self.vocab)

    def decode(self, ids, **kw):
        inv = {v: k.replace("Ġ", " ") for k, v in self.vocab.items()}
        return "".join(inv.get(i, "?") for i in ids)

    def apply_chat_template(self, msgs, tokenize=False, add_generation_prompt=True, **kw):
        return "<|user|>\n" + msgs[0]["content"] + "\n<|assistant|>\n"


class MockBackend(Backend):
    """Deterministic fake model with controllable accuracy. Exercises every pipeline code path.

    answer_key maps question text -> gold letter. Outputs depend only on hashes of the question,
    the request seed and `seed`, so runs are reproducible.
    """
    name = "mock"
    _QRE = re.compile(r"Question: (.*?)\nA\. ", re.S)

    def __init__(self, answer_key: dict, accuracy: float = 0.65, seed: int = 0, topk: int = 20,
                 nan_question: str | None = None):
        self.tokenizer = _MockTokenizer()
        self.letter_variants, self.letter_variant_strings = find_letter_variants(self.tokenizer)
        self.answer_key, self.accuracy, self.seed, self.topk = answer_key, accuracy, seed, topk
        self.nan_question = nan_question

    def _latent(self, prompt):
        m = self._QRE.search(prompt)
        q = m.group(1).strip() if m else prompt
        gold = self.answer_key.get(q, "A")
        correct = _u(self.seed, q, "c") < self.accuracy
        wrong = [L for L in LETTERS if L != gold]
        intended = gold if correct else wrong[int(_u(self.seed, q, "w") * 3)]
        s = min(1.0, max(0.0, (0.55 if correct else 0.30) + 0.45 * _u(self.seed, q, "s")))
        return q, intended, s

    def _reasoning(self, q, n_words):
        return " ".join(f"step{i}" for i in range(n_words))

    def _greedy(self, prompt, q, X, s, max_new):
        kind = int(_u(self.seed, q, "kind") * 25)
        conf = int(round((60 + 38 * s) / 5) * 5)
        body = "Let me think step by step. " + self._reasoning(q, 20 + int(80 * _u(self.seed, q, "len")))
        if kind == 0:   # no "Answer:" line, fallback phrase, no confidence -> forced prefix + recovery
            return GenOutput(body + f"\nSo the answer is {X}.", 60, "stop")
        if kind == 1:   # truncated, nothing parsable
            return GenOutput(body, max_new, "length")
        if kind == 2:
            tail = f"\n**Answer:** {X}\n**Confidence:** {conf}%"
        elif kind == 3:
            tail = f"\nAnswer: ({X})\nConfidence: {conf / 100:.2f}"
        elif kind == 4:
            tail = f"\nAnswer: {X}\n"
        else:
            tail = f"\nAnswer: {X}\nConfidence: {conf}"
        return GenOutput(body + tail, 40 + len(body.split()), "stop")

    def _sample(self, q, X, s, seed, j):
        u = _u(self.seed, q, seed, j)
        if u < 0.03:
            return GenOutput(self._reasoning(q, 30), 1024, "length")
        if u < 0.06:
            return GenOutput("I am not sure.", 5, "stop")
        if _u(self.seed, q, seed, j, "agree") < 0.45 + 0.55 * s:
            L = X
        else:
            L = LETTERS[int(_u(self.seed, q, seed, j, "o") * 4)]
        return GenOutput(f"Reasoning.\nAnswer: {L}\nConfidence: {int(50 + 50 * s)}", 30, "stop")

    def generate(self, reqs):
        res = []
        for r in reqs:
            q, X, s = self._latent(r.prompt)
            if r.prompt.rstrip().endswith("Confidence:"):
                txt = f" {int(round(55 + 40 * s))}" if _u(self.seed, q, "cfail") > 0.02 else " unsure"
                res.append([GenOutput(txt, 2, "stop")])
            elif r.temperature == 0:
                res.append([self._greedy(r.prompt, q, X, s, r.max_new_tokens)])
            else:
                res.append([self._sample(q, X, s, r.seed, j) for j in range(r.n)])
        return res

    def next_token_logprobs(self, prompts):
        res = []
        for p in prompts:
            q, X, s = self._latent(p)
            if self.nan_question is not None and q == self.nan_question:
                check_finite([math.nan], "mock logprobs")
            cot = "Think through the problem" in p
            top = X
            if cot and _u(self.seed, q, "mismatch") < 0.05:  # argmax differs from written letter
                top = LETTERS[(LETTERS.index(X) + 1) % 4]
            mass = 0.55 + 0.44 * _u(self.seed, q, "mass", cot)
            ptop = 0.30 + 0.69 * s
            rest = [L for L in LETTERS if L != top]
            w = [_u(self.seed, q, L, cot) + 0.05 for L in rest]
            probs = {top: ptop}
            probs.update({L: (1 - ptop) * wi / sum(w) for L, wi in zip(rest, w)})
            drop = rest[0] if _u(self.seed, q, "drop", cot) < 0.10 else None  # missing from top-k
            out = {}
            for L in LETTERS:
                if L == drop:
                    continue
                sp, nosp = self.letter_variants[L][1], self.letter_variants[L][0]
                out[sp] = math.log(mass * probs[L] * 0.9)
                out[nosp] = math.log(mass * probs[L] * 0.1)
            res.append(out)
        return res


# ------------------------------------------------------------------------------------- smoke test
def smoke_test(model_path: str, backend: str = "vllm") -> dict:
    """Load the backend, generate a few tokens and read letter logprobs. Run in a subprocess so a failed
    engine start cannot leave the notebook kernel holding GPU memory."""
    import json
    import time

    from config import Config

    cfg = Config()
    t0 = time.time()
    be = VLLMBackend(model_path, cfg) if backend == "vllm" else HFBackend(model_path, cfg)
    load_s = time.time() - t0
    out = be.generate([GenRequest(be.render("Reply with the single word OK."), 8)])[0][0]
    lp = be.next_token_logprobs([be.render("Which letter comes first in the alphabet? Reply 'Answer: <letter>'.")
                                 + "Answer:"])[0]
    res = {"ok": True, "load_seconds": load_s, "text": out.text, "letter_logprobs": {str(k): v for k, v in lp.items()},
           "info": be.info()}
    print("SMOKE_RESULT " + json.dumps(res, default=str))
    return res


if __name__ == "__main__":
    import sys as _sys
    smoke_test(_sys.argv[1], _sys.argv[2] if len(_sys.argv) > 2 else "vllm")


In [ ]:
%%writefile medqa_src/medqa_conf/generate.py
"""Model calls 1-6 per question, chunked, with JSONL checkpointing, resume and a time budget."""
from __future__ import annotations

import glob
import gzip
import json
import math
import os
import time

from . import parsing, prompts
from .backends import GenRequest

LETTERS = ("A", "B", "C", "D")


class ResumeMismatchError(RuntimeError):
    pass


def letter_distribution(logprobs: dict, letter_variants: dict) -> dict:
    """Sum the probability of the 'L' and ' L' variants, then renormalise over A-D."""
    raw, missing = {}, []
    for L in LETTERS:
        present = [logprobs[t] for t in letter_variants[L] if t in logprobs]
        if not present:
            missing.append(L)
        raw[L] = sum(math.exp(v) for v in present)
    mass = sum(raw.values())
    if mass > 0:
        p = {L: raw[L] / mass for L in LETTERS}
    else:
        p = {L: 0.25 for L in LETTERS}
    best = max(LETTERS, key=lambda L: (p[L], -LETTERS.index(L)))  # ties -> alphabetical
    return {"p": p, "letter_mass": mass, "argmax": best, "missing_letters": missing}


def _gen_dict(o):
    return {"text": o.text, "tokens": o.n_tokens, "finish": o.finish_reason}


def process_chunk(backend, cfg, qs) -> list[dict]:
    """Run calls 1-6 for a list of Questions; returns raw records (one per question)."""
    cot_prompts = [backend.render(prompts.cot_message(q)) for q in qs]
    dir_prompts = [backend.render(prompts.direct_message(q)) for q in qs]
    reqs = []
    for q, cp in zip(qs, cot_prompts):
        reqs.append(GenRequest(cp, cfg.greedy_max_new_tokens, temperature=0.0))
    for q, cp in zip(qs, cot_prompts):
        reqs.append(GenRequest(cp, cfg.sample_max_new_tokens, temperature=cfg.sample_temperature,
                               top_p=cfg.sample_top_p, n=cfg.n_samples, seed=cfg.base_seed + q.qid))
    outs = backend.generate(reqs)
    greedy = [o[0] for o in outs[: len(qs)]]
    samples = outs[len(qs):]
    for s in samples:
        if len(s) != cfg.n_samples:
            raise RuntimeError(f"expected {cfg.n_samples} samples, got {len(s)}")

    # calls 3 and 5: next-token letter distributions
    cot_prefixes = [prompts.cot_letter_prefix(g.text) for g in greedy]
    lp_prompts = [cp + pre for cp, (pre, _) in zip(cot_prompts, cot_prefixes)]
    lp_prompts += [dp + prompts.DIRECT_LETTER_PREFILL for dp in dir_prompts]
    lps = backend.next_token_logprobs(lp_prompts)
    cot_lp, dir_lp = lps[: len(qs)], lps[len(qs):]
    cot_dist = [letter_distribution(lp, backend.letter_variants) for lp in cot_lp]
    dir_dist = [letter_distribution(lp, backend.letter_variants) for lp in dir_lp]

    # calls 4 (only if needed) and 6
    conf_reqs, conf_owner = [], []
    for i, (g, (pre, _)) in enumerate(zip(greedy, cot_prefixes)):
        c, _f = parsing.parse_confidence(g.text)
        if math.isnan(c):
            conf_reqs.append(GenRequest(cot_prompts[i] + prompts.cot_recovery_prefill(pre, cot_dist[i]["argmax"]),
                                        cfg.confidence_max_new_tokens, temperature=0.0))
            conf_owner.append(("cot", i))
    for i, dp in enumerate(dir_prompts):
        conf_reqs.append(GenRequest(dp + prompts.direct_conf_prefill(dir_dist[i]["argmax"]),
                                    cfg.confidence_max_new_tokens, temperature=0.0))
        conf_owner.append(("direct", i))
    conf_outs = backend.generate(conf_reqs) if conf_reqs else []
    recovery = {i: None for i in range(len(qs))}
    dconf = {}
    for (kind, i), o in zip(conf_owner, conf_outs):
        (recovery if kind == "cot" else dconf)[i] = _gen_dict(o[0])

    recs = []
    for i, q in enumerate(qs):
        recs.append({
            "qid": q.qid, "qhash": q.qhash, "meta_info": q.meta_info, "gold": q.gold,
            "cot": {"greedy": _gen_dict(greedy[i]),
                    "samples": [_gen_dict(o) for o in samples[i]],
                    "forced_answer": cot_prefixes[i][1],
                    "logprobs": {str(k): v for k, v in cot_lp[i].items()},
                    "recovery": recovery[i]},
            "direct": {"logprobs": {str(k): v for k, v in dir_lp[i].items()},
                       "confidence_gen": dconf[i]},
        })
    return recs


# ----------------------------------------------------------------------------------- checkpointing
def checkpoint_header(cfg, backend_info: dict, model_revision: str | None) -> dict:
    return {"type": "header", "model_id": cfg.model_id, "config_hash": cfg.config_hash(),
            "model_revision": model_revision, "backend": backend_info.get("backend")}


def read_checkpoint(path: str):
    header, recs = None, {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:  # torn last line from a killed session
                continue
            if d.get("type") == "header":
                header = d
            else:
                recs[d["qid"]] = d
    return header, recs


def find_resume_checkpoints(cfg, search_roots) -> list[str]:
    found = []
    for root in search_roots:
        if root and os.path.isdir(root):
            found += glob.glob(os.path.join(root, "**", cfg.model_slug, "checkpoint.jsonl"), recursive=True)
    return sorted(set(found))


def load_resume(cfg, header: dict, paths, log=print) -> dict:
    """Records from previous checkpoints with the same model + config hash. Mismatched revision or
    backend raises (mixing outputs from different weights/engines is not allowed)."""
    recs = {}
    for p in paths:
        h, r = read_checkpoint(p)
        if h is None or h.get("model_id") != header["model_id"] or h.get("config_hash") != header["config_hash"]:
            log(f"[resume] ignoring {p}: different model or config hash")
            continue
        for key in ("model_revision", "backend"):
            if h.get(key) != header.get(key):
                raise ResumeMismatchError(
                    f"checkpoint {p} has {key}={h.get(key)!r} but this run has {header.get(key)!r}. "
                    "Detach that input (start fresh) or pin the same revision/backend.")
        log(f"[resume] {len(r)} finished questions from {p}")
        recs.update(r)
    return recs


def run_generation(backend, cfg, questions, out_dir, model_revision=None, resume_roots=(),
                   time_budget_s: float | None = None, t_start: float | None = None, log=print) -> dict:
    """Generate for all questions; checkpoint every cfg.checkpoint_every. Returns a status dict."""
    os.makedirs(out_dir, exist_ok=True)
    t_start = t_start if t_start is not None else time.time()
    ck_path = os.path.join(out_dir, "checkpoint.jsonl")
    header = checkpoint_header(cfg, backend.info(), model_revision)
    prev_paths = find_resume_checkpoints(cfg, list(resume_roots))
    if os.path.exists(ck_path):
        prev_paths.append(ck_path)
    done = load_resume(cfg, header, prev_paths, log=log)
    wanted = {q.qid for q in questions}
    done = {k: v for k, v in done.items() if k in wanted}
    with open(ck_path + ".tmp", "w", encoding="utf-8") as f:  # rewrite: header + resumed records
        f.write(json.dumps(header) + "\n")
        for k in sorted(done):
            f.write(json.dumps(done[k]) + "\n")
    os.replace(ck_path + ".tmp", ck_path)

    todo = [q for q in questions if q.qid not in done]
    log(f"[generate] {len(done)} resumed, {len(todo)} to do")
    chunk_times, n_new, stopped_early = [], 0, False
    for b in range(0, len(todo), cfg.checkpoint_every):
        if time_budget_s is not None and chunk_times:
            est = max(chunk_times)
            if time.time() - t_start + est > time_budget_s:
                log(f"[generate] time budget reached; stopping with {len(todo) - b} questions left")
                stopped_early = True
                break
        chunk = todo[b:b + cfg.checkpoint_every]
        t0 = time.time()
        recs = process_chunk(backend, cfg, chunk)
        with open(ck_path, "a", encoding="utf-8") as f:
            for r in recs:
                f.write(json.dumps(r) + "\n")
            f.flush()
            os.fsync(f.fileno())
        for r in recs:
            done[r["qid"]] = r
        dt = time.time() - t0
        chunk_times.append(dt)
        n_new += len(chunk)
        log(f"[generate] {len(done)}/{len(questions)} done ({dt:.1f}s for {len(chunk)} q)")
    gen_time = sum(chunk_times)
    return {"records": [done[k] for k in sorted(done)], "complete": len(done) == len(questions),
            "n_resumed": len(questions) - len(todo), "n_generated": n_new, "stopped_early": stopped_early,
            "generation_seconds": gen_time,
            "seconds_per_question": gen_time / n_new if n_new else None,
            "checkpoint": ck_path}


def determinism_check(backend, cfg, questions, records, n: int) -> dict:
    """Re-run CoT greedy on the first n questions; count byte-identical outputs."""
    qs = questions[:n]
    by_qid = {r["qid"]: r for r in records}
    reqs = [GenRequest(backend.render(prompts.cot_message(q)), cfg.greedy_max_new_tokens, temperature=0.0)
            for q in qs]
    outs = backend.generate(reqs)
    same = [o[0].text == by_qid[q.qid]["cot"]["greedy"]["text"] for q, o in zip(qs, outs)]
    return {"n": len(qs), "identical": int(sum(same)), "qids_differing": [q.qid for q, s in zip(qs, same) if not s]}


def write_raw_generations(records, path):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


In [ ]:
%%writefile medqa_src/medqa_conf/signals.py
"""Raw records -> per_question table (one row per qid x setting; no raw text)."""
from __future__ import annotations

import collections
import math

import pandas as pd

from . import parsing
from .generate import letter_distribution

LETTERS = ("A", "B", "C", "D")
COLUMNS = [
    "model", "setting", "qid", "qhash", "meta_info", "gold", "answer", "correct",
    "p_A", "p_B", "p_C", "p_D", "letter_mass", "n_letters_missing_topk",
    "conf_prob", "conf_stated", "conf_agree",
    "stated_recovered", "stated_fraction_flag", "stated_missing_before_recovery",
    "sample_answers", "majority_answer", "n_samples_parsed",
    "answer_parsed", "answer_parse_method", "forced_answer", "parsed_vs_argmax_mismatch",
    "greedy_tokens", "greedy_truncated", "n_samples_truncated",
]


def majority_vote(sample_letters, official):
    parsed = [s for s in sample_letters if s is not None]
    if not parsed:
        return official
    cnt = collections.Counter(parsed)
    top = max(cnt.values())
    tied = sorted(L for L, c in cnt.items() if c == top)
    return official if official in tied else tied[0]


def _int_keys(d):
    return {int(k): v for k, v in d.items()}


def records_to_frame(records, model_id: str, letter_variants: dict) -> pd.DataFrame:
    rows = []
    for r in records:
        cot = r["cot"]
        sample_letters = [parsing.parse_answer(s["text"])[0] for s in cot["samples"]]
        sample_str = "".join(L if L is not None else "-" for L in sample_letters)
        n_trunc = sum(s["finish"] == "length" for s in cot["samples"])
        g = cot["greedy"]
        written, method = parsing.parse_answer(g["text"])
        common = dict(model=model_id, qid=r["qid"], qhash=r["qhash"], meta_info=r["meta_info"], gold=r["gold"],
                      sample_answers=sample_str, n_samples_parsed=sum(L is not None for L in sample_letters),
                      n_samples_truncated=n_trunc)

        # --- CoT
        d = letter_distribution(_int_keys(cot["logprobs"]), letter_variants)
        X = d["argmax"]
        c, frac = parsing.parse_confidence(g["text"])
        missing_before = int(math.isnan(c))
        recovered = 0
        if math.isnan(c) and cot.get("recovery") is not None:
            c, frac = parsing.parse_confidence_continuation(cot["recovery"]["text"])
            recovered = int(not math.isnan(c))
        rows.append(dict(common, setting="cot", **_letter_cols(d), answer=X, correct=int(X == r["gold"]),
                         conf_prob=d["p"][X], conf_stated=c,
                         conf_agree=sum(L == X for L in sample_letters) / len(sample_letters),
                         stated_recovered=recovered, stated_fraction_flag=int(frac),
                         stated_missing_before_recovery=missing_before,
                         majority_answer=majority_vote(sample_letters, X),
                         answer_parsed=written if written else "", answer_parse_method=method,
                         forced_answer=int(cot["forced_answer"]),
                         parsed_vs_argmax_mismatch=(int(written != X) if written else math.nan),
                         greedy_tokens=g["tokens"], greedy_truncated=int(g["finish"] == "length")))

        # --- Direct
        dr = r["direct"]
        d = letter_distribution(_int_keys(dr["logprobs"]), letter_variants)
        X = d["argmax"]
        cg = dr.get("confidence_gen")
        c, frac = parsing.parse_confidence_continuation(cg["text"] if cg else None)
        rows.append(dict(common, setting="direct", **_letter_cols(d), answer=X, correct=int(X == r["gold"]),
                         conf_prob=d["p"][X], conf_stated=c,
                         conf_agree=sum(L == X for L in sample_letters) / len(sample_letters),
                         stated_recovered=0, stated_fraction_flag=int(frac),
                         stated_missing_before_recovery=int(math.isnan(c)),
                         majority_answer=majority_vote(sample_letters, X),
                         answer_parsed="", answer_parse_method="", forced_answer=math.nan,
                         parsed_vs_argmax_mismatch=math.nan, greedy_tokens=math.nan, greedy_truncated=math.nan))
    df = pd.DataFrame(rows)[COLUMNS]
    return df.sort_values(["setting", "qid"], kind="stable").reset_index(drop=True)


def _letter_cols(d):
    out = {f"p_{L}": d["p"][L] for L in LETTERS}
    out["letter_mass"] = d["letter_mass"]
    out["n_letters_missing_topk"] = len(d["missing_letters"])
    return out


In [ ]:
%%writefile medqa_src/medqa_conf/metrics.py
"""Metrics (PREREGISTRATION.md section 7). All functions take numpy arrays: conf in [0,1], y in {0,1}."""
from __future__ import annotations

import math

import numpy as np
from scipy.stats import norm, rankdata


def accuracy(y) -> float:
    return float(np.mean(y))


def wilson_ci(k: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    if n == 0:
        return (math.nan, math.nan)
    z = norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)


def auroc(conf, y) -> float:
    """Mann-Whitney AUROC, ties count 1/2 (== sklearn.metrics.roc_auc_score)."""
    conf, y = np.asarray(conf, float), np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return math.nan
    r = rankdata(conf)  # average ranks for ties
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def equal_width_edges(n_bins: int) -> np.ndarray:
    # i / n (true division) so that edges like 12/15 equal 0.8 == 4/5 exactly (conf_agree takes k/5 values)
    return np.array([i / n_bins for i in range(n_bins + 1)])


def bin_index(conf, edges) -> np.ndarray:
    """Left-closed bins [e_i, e_{i+1}); the last bin also includes the right edge."""
    conf, edges = np.asarray(conf, float), np.asarray(edges, float)
    if np.any(conf < edges[0]) or np.any(conf > edges[-1]):
        raise ValueError("confidence outside the bin edges")
    idx = np.searchsorted(edges, conf, side="right") - 1
    return np.minimum(idx, len(edges) - 2)


def ece(conf, y, n_bins: int = 15, edges=None) -> float:
    conf, y = np.asarray(conf, float), np.asarray(y, float)
    edges = equal_width_edges(n_bins) if edges is None else np.asarray(edges, float)
    idx = bin_index(conf, edges)
    nb = len(edges) - 1
    cnt = np.bincount(idx, minlength=nb)
    sc = np.bincount(idx, weights=conf, minlength=nb)
    sy = np.bincount(idx, weights=y, minlength=nb)
    m = cnt > 0
    return float(np.sum(np.abs(sy[m] - sc[m])) / len(conf))


def reliability_bins(conf, y, n_bins: int = 15, edges=None) -> dict:
    conf, y = np.asarray(conf, float), np.asarray(y, float)
    edges = equal_width_edges(n_bins) if edges is None else np.asarray(edges, float)
    idx = bin_index(conf, edges)
    nb = len(edges) - 1
    cnt = np.bincount(idx, minlength=nb)
    with np.errstate(invalid="ignore", divide="ignore"):
        mc = np.bincount(idx, weights=conf, minlength=nb) / cnt
        acc = np.bincount(idx, weights=y, minlength=nb) / cnt
    return {"edges": edges.tolist(), "count": cnt.tolist(), "mean_conf": mc.tolist(), "accuracy": acc.tolist()}


def adaptive_ece(conf, y, n_bins: int = 15) -> float:
    """Equal-mass bins: stable sort by confidence, split into n_bins contiguous groups (np.array_split)."""
    conf, y = np.asarray(conf, float), np.asarray(y, float)
    order = np.argsort(conf, kind="stable")
    tot = 0.0
    for g in np.array_split(order, n_bins):
        if len(g):
            tot += abs(y[g].sum() - conf[g].sum())
    return float(tot / len(conf))


def brier(conf, y) -> float:
    conf, y = np.asarray(conf, float), np.asarray(y, float)
    return float(np.mean((conf - y) ** 2))


def overconfidence_gap(conf, y) -> float:
    return float(np.mean(conf) - np.mean(y))


def expected_correct_curve(conf, y) -> np.ndarray:
    """E[#correct among the top-k] for k = 1..N under random tie-breaking (exact)."""
    conf, y = np.asarray(conf, float), np.asarray(y, float)
    n = len(conf)
    order = np.argsort(-conf, kind="stable")
    cs, ys = conf[order], y[order]
    new_group = np.ones(n, bool)
    new_group[1:] = cs[1:] != cs[:-1]
    gid = np.cumsum(new_group) - 1
    starts = np.flatnonzero(new_group)
    sizes = np.diff(np.append(starts, n))
    gcorr = np.bincount(gid, weights=ys)
    corr_before = np.concatenate([[0.0], np.cumsum(gcorr)[:-1]])
    k = np.arange(1, n + 1)
    g = gid[k - 1]
    return corr_before[g] + (k - starts[g]) * gcorr[g] / sizes[g]


def coverage_k(n: int, coverage: float) -> int:
    return int(math.floor(round(coverage * n, 9)))


def selective_accuracy(conf, y, coverage: float = 0.8) -> float:
    n = len(conf)
    k = coverage_k(n, coverage)
    if k == 0:
        return math.nan
    return float(expected_correct_curve(conf, y)[k - 1] / k)


def oracle_selective_accuracy(y, coverage: float = 0.8) -> float:
    k = coverage_k(len(y), coverage)
    return float(min(int(np.sum(y)), k) / k)


def risk_coverage(conf, y) -> tuple[np.ndarray, np.ndarray]:
    n = len(conf)
    k = np.arange(1, n + 1)
    return k / n, 1.0 - expected_correct_curve(conf, y) / k


def aurc(conf, y) -> float:
    return float(np.mean(risk_coverage(conf, y)[1]))


def signal_metrics(conf, y, n_bins: int = 15, coverage: float = 0.8) -> dict:
    n = len(conf)
    curve = expected_correct_curve(conf, y)  # shared by selective accuracy and AURC
    k = coverage_k(n, coverage)
    return {
        "auroc": auroc(conf, y),
        "ece": ece(conf, y, n_bins),
        "adaptive_ece": adaptive_ece(conf, y, n_bins),
        "brier": brier(conf, y),
        "mean_conf": float(np.mean(conf)),
        "overconfidence_gap": overconfidence_gap(conf, y),
        "selective_acc": float(curve[k - 1] / k) if k else math.nan,
        "aurc": float(np.mean(1.0 - curve / np.arange(1, n + 1))),
    }


METRIC_NAMES = ("auroc", "ece", "adaptive_ece", "brier", "mean_conf", "overconfidence_gap",
                "selective_acc", "aurc")


In [ ]:
%%writefile medqa_src/medqa_conf/stats.py
"""Paired bootstrap, fast DeLong (Sun & Xu 2014), Holm correction, cross-fitted Platt scaling."""
from __future__ import annotations

import math

import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm, rankdata

from . import metrics as M


# ------------------------------------------------------------------------------------------ DeLong
def _fast_delong(preds_sorted: np.ndarray, m: int):
    """preds_sorted: (k, n) with the m positives first. Returns (aucs, covariance)."""
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.vstack([rankdata(pos[r]) for r in range(k)])
    ty = np.vstack([rankdata(neg[r]) for r in range(k)])
    tz = np.vstack([rankdata(preds_sorted[r]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.atleast_2d(np.cov(v01))
    sy = np.atleast_2d(np.cov(v10))
    return aucs, sx / m + sy / n


def delong_paired(y, s1, s2) -> dict:
    """Two-sided paired DeLong test of AUROC(s1) - AUROC(s2)."""
    y = np.asarray(y)
    order = np.concatenate([np.flatnonzero(y == 1), np.flatnonzero(y == 0)])
    m = int((y == 1).sum())
    if m == 0 or m == len(y):
        return {"auc1": math.nan, "auc2": math.nan, "diff": math.nan, "se": math.nan, "z": math.nan, "p": math.nan}
    preds = np.vstack([np.asarray(s1, float)[order], np.asarray(s2, float)[order]])
    aucs, cov = _fast_delong(preds, m)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    diff = aucs[0] - aucs[1]
    if var <= 1e-15:
        z, p = 0.0 if abs(diff) < 1e-12 else math.copysign(math.inf, diff), 1.0 if abs(diff) < 1e-12 else 0.0
        se = 0.0
    else:
        se = math.sqrt(var)
        z = diff / se
        p = 2 * norm.sf(abs(z))
    return {"auc1": float(aucs[0]), "auc2": float(aucs[1]), "diff": float(diff), "se": float(se),
            "z": float(z), "p": float(p)}


# -------------------------------------------------------------------------------------------- Holm
def holm(pvals) -> list[float]:
    p = np.asarray(pvals, float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    idx = np.flatnonzero(ok)
    if len(idx) == 0:
        return out.tolist()
    order = idx[np.argsort(p[idx], kind="stable")]
    m = len(order)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[i]))
        out[i] = running
    return out.tolist()


# --------------------------------------------------------------------------------------- bootstrap
def bootstrap_indices(n: int, n_boot: int, seed: int) -> np.ndarray:
    return np.random.default_rng(seed).integers(0, n, size=(n_boot, n))


def paired_bootstrap(y, signals: dict, n_boot: int, seed: int, n_bins: int, coverage: float) -> dict:
    """Bootstrap replicates of every metric for every signal, sharing indices. Returns
    {"accuracy": array(B), signal: {metric: array(B)}}."""
    y = np.asarray(y, float)
    idx = bootstrap_indices(len(y), n_boot, seed)
    out = {"accuracy": np.empty(n_boot)}
    for s in signals:
        out[s] = {m: np.empty(n_boot) for m in M.METRIC_NAMES}
    for b in range(n_boot):
        ib = idx[b]
        yb = y[ib]
        out["accuracy"][b] = yb.mean()
        for s, conf in signals.items():
            r = M.signal_metrics(np.asarray(conf, float)[ib], yb, n_bins, coverage)
            for m in M.METRIC_NAMES:
                out[s][m][b] = r[m]
    return out


def percentile_ci(x, alpha: float = 0.05) -> list:
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return [None, None]
    return [float(np.percentile(x, 100 * alpha / 2)), float(np.percentile(x, 100 * (1 - alpha / 2)))]


# ------------------------------------------------------------------------------------------- Platt
def platt_fit(s, y) -> tuple[float, float]:
    s, y = np.asarray(s, float), np.asarray(y, float)

    def nll(w):
        z = w[0] * s + w[1]
        return float(np.sum(np.logaddexp(0, z) - y * z))

    def grad(w):
        p = 1 / (1 + np.exp(-(w[0] * s + w[1])))
        return np.array([np.sum((p - y) * s), np.sum(p - y)])

    res = minimize(nll, np.array([1.0, 0.0]), jac=grad, method="BFGS")
    return float(res.x[0]), float(res.x[1])


def crossfit_platt(s, y, folds: int, seed: int) -> np.ndarray:
    s, y = np.asarray(s, float), np.asarray(y, float)
    perm = np.random.default_rng(seed).permutation(len(s))
    out = np.empty(len(s))
    for test in np.array_split(perm, folds):
        train = np.setdiff1d(perm, test)
        if len(np.unique(y[train])) < 2:
            out[test] = y[train].mean()
            continue
        a, b = platt_fit(s[train], y[train])
        out[test] = 1 / (1 + np.exp(-(a * s[test] + b)))
    return out


In [ ]:
%%writefile medqa_src/medqa_conf/report.py
"""Analysis from per_question.csv only: summary.json, results_table.md, figures; plus multi-model combine.

    python -m medqa_conf.report --from-csv outputs/<slug>/per_question.csv [--out DIR]
    python -m medqa_conf.report --combine DIR_OR_CSV [DIR_OR_CSV ...] --out DIR
"""
from __future__ import annotations

import argparse
import glob
import json
import math
import os
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata

from . import metrics as M
from . import stats as S

SIGNALS = ("stated", "prob", "agree")
SIGNAL_COLS = {"stated": "conf_stated", "prob": "conf_prob", "agree": "conf_agree"}
PAIRS = (("stated", "prob"), ("stated", "agree"), ("prob", "agree"))
SETTINGS = ("cot", "direct")
SETTING_SEED_OFFSET = {"cot": 0, "direct": 1}


def default_config():
    try:
        from config import Config
    except ImportError:
        root = Path(__file__).resolve().parents[2]
        if (root / "config.py").exists():
            sys.path.insert(0, str(root))
        from config import Config
    return Config()


def analysis_params(cfg) -> dict:
    from config import ANALYSIS_KEYS
    return {k: getattr(cfg, k) for k in ANALYSIS_KEYS}


def read_csv(path) -> pd.DataFrame:
    return pd.read_csv(path, dtype={"sample_answers": str, "answer": str, "gold": str, "majority_answer": str,
                                    "answer_parsed": str, "answer_parse_method": str, "meta_info": str,
                                    "qhash": str, "model": str, "setting": str},
                       keep_default_na=False, na_values=[""])


def _pair_key(a, b):
    return f"{a}_vs_{b}"


def _val_ci(v, boot, alpha):
    return {"value": v, "ci": S.percentile_ci(boot, alpha)}


# --------------------------------------------------------------------------------------- core block
def analyze_block(y, signals: dict, ap: dict, seed: int) -> dict:
    """Point estimates, bootstrap CIs, pairwise differences and DeLong for one set of rows."""
    y = np.asarray(y, int)
    n, alpha = len(y), ap["alpha"]
    if n == 0:
        return _empty_block(signals)
    boot = S.paired_bootstrap(y, signals, ap["n_boot"], seed, ap["n_bins"], ap["coverage"])
    acc = M.accuracy(y)
    out = {
        "n": n,
        "accuracy": {"value": acc, "wilson_ci": list(M.wilson_ci(int(y.sum()), n, alpha)),
                     "bootstrap_ci": S.percentile_ci(boot["accuracy"], alpha)},
        "references": {"k": M.coverage_k(n, ap["coverage"]), "random_selective_acc": acc,
                       "oracle_selective_acc": M.oracle_selective_accuracy(y, ap["coverage"])},
        "signals": {}, "pairs": {},
    }
    point = {}
    for s, conf in signals.items():
        point[s] = M.signal_metrics(conf, y, ap["n_bins"], ap["coverage"])
        vals = np.asarray(conf, float)
        uniq, cnt = np.unique(vals, return_counts=True)
        out["signals"][s] = {m: _val_ci(point[s][m], boot[s][m], alpha) for m in M.METRIC_NAMES}
        out["signals"][s]["frac_tied"] = float(cnt[cnt > 1].sum() / n)
        out["signals"][s]["n_unique"] = int(len(uniq))
    names = list(signals)
    pairs = [(a, b) for a, b in PAIRS if a in names and b in names]
    for a, b in pairs:
        d = {m: _val_ci(point[a][m] - point[b][m], boot[a][m] - boot[b][m], alpha) for m in M.METRIC_NAMES}
        d["delong"] = S.delong_paired(y, signals[a], signals[b])
        out["pairs"][_pair_key(a, b)] = d
    p_holm = S.holm([out["pairs"][_pair_key(a, b)]["delong"]["p"] for a, b in pairs])
    for (a, b), ph in zip(pairs, p_holm):
        pr = out["pairs"][_pair_key(a, b)]
        pr["p_holm_within_model"] = ph
        pr["supported_within_model"] = _supported(ph, pr["auroc"]["ci"], alpha)
    return out


def _empty_block(signals) -> dict:
    """Same structure as analyze_block, all values missing (no usable rows)."""
    na = {"value": math.nan, "ci": [None, None]}
    names = list(signals)
    out = {"n": 0, "accuracy": {"value": math.nan, "wilson_ci": [None, None], "bootstrap_ci": [None, None]},
           "references": {"k": 0, "random_selective_acc": math.nan, "oracle_selective_acc": math.nan},
           "signals": {s: dict({m: dict(na) for m in M.METRIC_NAMES}, frac_tied=math.nan, n_unique=0) for s in names},
           "pairs": {}}
    for a, b in PAIRS:
        if a in names and b in names:
            out["pairs"][_pair_key(a, b)] = dict(
                {m: dict(na) for m in M.METRIC_NAMES},
                delong={"auc1": math.nan, "auc2": math.nan, "diff": math.nan, "se": math.nan, "z": math.nan,
                        "p": math.nan}, p_holm_within_model=math.nan, supported_within_model=False)
    return out


def _supported(p_holm, ci, alpha):
    if p_holm is None or (isinstance(p_holm, float) and math.isnan(p_holm)) or ci[0] is None:
        return False
    return bool(p_holm < alpha and (ci[0] > 0 or ci[1] < 0))


def _direction(d):
    if d is None or math.isnan(d):
        return "NA"
    return "prob>agree" if d > 0 else ("agree>prob" if d < 0 else "equal")


def hypotheses(block: dict, holm_key: str = "p_holm_within_model", alpha: float = 0.05) -> dict:
    P = block["pairs"]

    def sup(pair, sign=None):
        pr = P[pair]
        ok = _supported(pr[holm_key], pr["auroc"]["ci"], alpha)
        if sign is not None:
            ok = ok and np.sign(pr["auroc"]["value"]) == sign
        return bool(ok)

    st = block["signals"]["stated"]
    gap_lo = st["overconfidence_gap"]["ci"][0]
    e1, e2 = P["stated_vs_prob"]["ece"]["ci"][0], P["stated_vs_agree"]["ece"]["ci"][0]
    return {
        "H1": {"stated_below_prob": sup("stated_vs_prob", -1), "stated_below_agree": sup("stated_vs_agree", -1),
               "supported": sup("stated_vs_prob", -1) and sup("stated_vs_agree", -1)},
        "H2": {"overconfident": bool(gap_lo is not None and gap_lo > 0),
               "ece_above_prob": bool(e1 is not None and e1 > 0), "ece_above_agree": bool(e2 is not None and e2 > 0),
               "supported": bool(gap_lo is not None and gap_lo > 0 and e1 is not None and e1 > 0
                                 and e2 is not None and e2 > 0)},
        "H3": {"prob_vs_agree_differ": sup("prob_vs_agree"), "supported": sup("prob_vs_agree"),
               "direction": _direction(P["prob_vs_agree"]["auroc"]["value"])},
    }


# ---------------------------------------------------------------------------------------- validity
def validity(df_model: pd.DataFrame, setting: str, ap: dict) -> dict:
    cot = df_model[df_model.setting == "cot"]
    ds = df_model[df_model.setting == setting]
    vals = {
        "accuracy": float(ds.correct.mean()),
        "greedy_parse_fail_rate": float((cot.answer_parse_method == "none").mean()),
        "median_letter_mass": float(ds.letter_mass.median()),
        "sample_truncation_rate": float(cot.n_samples_truncated.sum() / (5 * len(cot))) if len(cot) else math.nan,
    }
    flags = {
        "accuracy_below_min": vals["accuracy"] < ap["flag_min_accuracy"],
        "parse_fail_above_max": vals["greedy_parse_fail_rate"] > ap["flag_max_parse_fail"],
        "letter_mass_below_min": vals["median_letter_mass"] < ap["flag_min_median_letter_mass"],
        "truncation_above_max": vals["sample_truncation_rate"] > ap["flag_max_truncation"],
    }
    return {"values": vals, "flags": {k: bool(v) for k, v in flags.items()}, "flagged": bool(any(flags.values()))}


# ------------------------------------------------------------------------------------- exploratory
def _entropy(sample_str: str) -> float:
    from collections import Counter
    c = np.array(list(Counter(sample_str).values()), float)
    p = c / c.sum()
    return float(-(p * np.log(p)).sum())


def exploratory(cc: pd.DataFrame, ap: dict) -> dict:
    y = cc.correct.to_numpy(int)
    n = len(y)
    if n == 0:
        return {"label": "EXPLORATORY - not part of the pre-registered conclusions", "n": 0,
                "majority_vote_accuracy": {"value": None, "wilson_ci": [None, None]},
                "neg_sample_entropy_auroc": None, "rank_average_prob_agree_auroc": None,
                "platt_crossfit_ece": {}, "by_meta_info": {}}
    maj =(cc.majority_answer == cc.gold).to_numpy(int)
    ent = cc.sample_answers.map(_entropy).to_numpy(float)
    rank_avg = (rankdata(cc.conf_prob) + rankdata(cc.conf_agree)) / 2
    out = {
        "label": "EXPLORATORY - not part of the pre-registered conclusions",
        "majority_vote_accuracy": {"value": float(maj.mean()), "wilson_ci": list(M.wilson_ci(int(maj.sum()), n))},
        "neg_sample_entropy_auroc": M.auroc(-ent, y),
        "rank_average_prob_agree_auroc": M.auroc(rank_avg, y),
        "platt_crossfit_ece": {}, "by_meta_info": {},
    }
    for s in SIGNALS:
        cal = S.crossfit_platt(cc[SIGNAL_COLS[s]].to_numpy(float), y, ap["cv_folds"], ap["cv_seed"])
        out["platt_crossfit_ece"][s] = M.ece(cal, y, ap["n_bins"])
    for mi, g in cc.groupby("meta_info", sort=True):
        yg = g.correct.to_numpy(int)
        out["by_meta_info"][str(mi)] = {
            "n": int(len(g)), "accuracy": float(yg.mean()),
            "auroc": {s: M.auroc(g[SIGNAL_COLS[s]].to_numpy(float), yg) for s in SIGNALS},
            "ece": {s: M.ece(g[SIGNAL_COLS[s]].to_numpy(float), yg, ap["n_bins"]) for s in SIGNALS},
        }
    return out


# ------------------------------------------------------------------------------------------ driver
def analyze_frame(df: pd.DataFrame, ap: dict) -> dict:
    models = sorted(df.model.unique())
    if len(models) != 1:
        raise ValueError(f"expected one model per CSV, got {models}")
    summary = {"model": models[0], "analysis_params": ap, "n_rows": int(len(df)), "settings": {}}
    for setting in SETTINGS:
        ds = df[df.setting == setting].sort_values("qid", kind="stable")
        if not len(ds):
            continue
        seed = ap["boot_seed"] + SETTING_SEED_OFFSET[setting]
        sig_cols = [SIGNAL_COLS[s] for s in SIGNALS]
        mask = ds[sig_cols].notna().all(axis=1)
        cc = ds[mask]
        block = analyze_block(cc.correct, {s: cc[SIGNAL_COLS[s]].to_numpy(float) for s in SIGNALS}, ap, seed)
        block["n_total"] = int(len(ds))
        block["n_complete_case"] = int(mask.sum())
        block["n_dropped"] = int((~mask).sum())
        block["n_missing"] = {s: int(ds[SIGNAL_COLS[s]].isna().sum()) for s in SIGNALS}
        block["hypotheses"] = hypotheses(block, alpha=ap["alpha"])
        block["validity"] = validity(df, setting, ap)
        # sensitivity: impute missing conf_stated with its median
        imp = ds.copy()
        med = float(imp.conf_stated.median()) if imp.conf_stated.notna().any() else 0.5
        imp["conf_stated"] = imp.conf_stated.fillna(med)
        m2 = imp[sig_cols].notna().all(axis=1)
        imp = imp[m2]
        sens = analyze_block(imp.correct, {s: imp[SIGNAL_COLS[s]].to_numpy(float) for s in SIGNALS}, ap, seed)
        sens["imputed_value"] = med
        sens["n_imputed"] = int(ds.conf_stated.isna().sum())
        sens["hypotheses"] = hypotheses(sens, alpha=ap["alpha"])
        block["sensitivity_median_impute"] = sens
        block["exploratory"] = exploratory(cc, ap)
        summary["settings"][setting] = block
    return summary


def clean_json(x):
    if isinstance(x, dict):
        return {str(k): clean_json(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [clean_json(v) for v in x]
    if isinstance(x, (np.bool_, bool)):
        return bool(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        x = float(x)
        return None if (math.isnan(x) or math.isinf(x)) else x
    return x


def dump_json(obj, path):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        json.dump(clean_json(obj), f, indent=2, sort_keys=True)
        f.write("\n")


# ------------------------------------------------------------------------------------------ tables
def _f(v, d=3):
    return "NA" if v is None or (isinstance(v, float) and math.isnan(v)) else f"{v:.{d}f}"


def _fci(x, d=3):
    ci = x.get("ci") or [None, None]
    return f"{_f(x['value'], d)} [{_f(ci[0], d)}, {_f(ci[1], d)}]"


def _p(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "NA"
    return f"{v:.2e}" if v < 1e-3 else f"{v:.4f}"


_METRIC_LABELS = {"auroc": "AUROC", "ece": "ECE", "adaptive_ece": "adaptive ECE", "brier": "Brier",
                  "mean_conf": "mean conf", "overconfidence_gap": "overconf. gap",
                  "selective_acc": "sel. acc @80%", "aurc": "AURC"}


def block_tables(b: dict, holm_key="p_holm_within_model", sup_key="supported_within_model") -> list[str]:
    L = []
    acc = b["accuracy"]
    L.append(f"N = {b['n']}; accuracy = {_f(acc['value'])} (Wilson 95% CI {_f(acc['wilson_ci'][0])}–"
             f"{_f(acc['wilson_ci'][1])}); selective-accuracy references at k = {b['references']['k']}: "
             f"random {_f(b['references']['random_selective_acc'])}, oracle {_f(b['references']['oracle_selective_acc'])}")
    L.append("")
    L.append("| signal | " + " | ".join(_METRIC_LABELS[m] for m in M.METRIC_NAMES) + " |")
    L.append("|---|---|---|---|---|---|---|---|---|")
    for s, v in b["signals"].items():
        L.append(f"| {s} | " + " | ".join(_fci(v[m]) for m in M.METRIC_NAMES) + " |")
    L.append("")
    L.append("| pair (a − b) | ΔAUROC [95% CI] | DeLong z | p | p (Holm) | supported | ΔECE | Δsel. acc |")
    L.append("|---|---|---|---|---|---|---|---|")
    for k, pr in b["pairs"].items():
        dl = pr["delong"]
        L.append(f"| {k} | {_fci(pr['auroc'])} | {_f(dl['z'], 2)} | {_p(dl['p'])} | {_p(pr.get(holm_key))} | "
                 f"{'yes' if pr.get(sup_key) else 'no'} | {_fci(pr['ece'])} | {_fci(pr['selective_acc'])} |")
    return L


def results_markdown(summary: dict) -> str:
    L = [f"# Results: {summary['model']}", "",
         "All numbers are computed by `medqa_conf.report` from `per_question.csv`. "
         "Intervals are paired-bootstrap percentile 95% CIs (2,000 resamples) unless marked Wilson.", ""]
    for setting, b in summary["settings"].items():
        tag = "primary, pre-registered" if setting == "cot" else "secondary"
        L += [f"## Setting: {setting} ({tag})", "",
              f"Complete-case N used: {b['n_complete_case']} of {b['n_total']} (dropped {b['n_dropped']}; "
              f"missing per signal: {b['n_missing']})", ""]
        v = b["validity"]
        L.append(f"**Validity flags:** {'FLAGGED' if v['flagged'] else 'none'} — "
                 + ", ".join(f"{k} = {_f(x)}" for k, x in v["values"].items())
                 + (" — " + ", ".join(k for k, f in v["flags"].items() if f) if v["flagged"] else ""))
        L.append("")
        L += block_tables(b)
        L.append("")
        h = b["hypotheses"]
        L += ["**Hypotheses (decision rule: Holm-adjusted DeLong p < 0.05 and bootstrap CI excludes 0):**", "",
              f"- H1 (stated AUROC below prob and agree): {'supported' if h['H1']['supported'] else 'not supported'} "
              f"(below prob: {h['H1']['stated_below_prob']}, below agree: {h['H1']['stated_below_agree']})",
              f"- H2 (stated overconfident and highest ECE): {'supported' if h['H2']['supported'] else 'not supported'} "
              f"(overconfident: {h['H2']['overconfident']}, ECE above prob: {h['H2']['ece_above_prob']}, "
              f"ECE above agree: {h['H2']['ece_above_agree']})",
              f"- H3 (prob and agree AUROC differ): {'supported' if h['H3']['supported'] else 'not supported'} "
              f"(observed direction: {h['H3']['direction']})", ""]
        s = b["sensitivity_median_impute"]
        L += [f"### Sensitivity: missing conf_stated imputed with median ({_f(s['imputed_value'])}; "
              f"{s['n_imputed']} imputed)", ""] + block_tables(s) + [""]
        e = b["exploratory"]
        L += ["### Exploratory (not part of the pre-registered conclusions)", "",
              f"- Majority-vote accuracy (5 CoT samples): {_f(e['majority_vote_accuracy']['value'])} "
              f"(Wilson {_f(e['majority_vote_accuracy']['wilson_ci'][0])}–{_f(e['majority_vote_accuracy']['wilson_ci'][1])})",
              f"- AUROC of −entropy of the 5 sample answers: {_f(e['neg_sample_entropy_auroc'])}",
              f"- AUROC of rank-average(prob, agree): {_f(e['rank_average_prob_agree_auroc'])}",
              "- ECE after 5-fold cross-fitted Platt scaling: "
              + ", ".join(f"{k} {_f(v)}" for k, v in e["platt_crossfit_ece"].items()), "",
              "| meta_info | n | accuracy | AUROC stated | AUROC prob | AUROC agree | ECE stated | ECE prob | ECE agree |",
              "|---|---|---|---|---|---|---|---|---|"]
        for mi, g in e["by_meta_info"].items():
            L.append(f"| {mi} | {g['n']} | {_f(g['accuracy'])} | " + " | ".join(_f(g["auroc"][s]) for s in SIGNALS)
                     + " | " + " | ".join(_f(g["ece"][s]) for s in SIGNALS) + " |")
        L.append("")
    return "\n".join(L) + "\n"


# ----------------------------------------------------------------------------------------- figures
def make_figures(df: pd.DataFrame, out_dir: str, ap: dict) -> list[str]:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs(out_dir, exist_ok=True)
    paths = []
    model = df.model.iloc[0]
    for setting in SETTINGS:
        ds = df[df.setting == setting].sort_values("qid", kind="stable")
        cc = ds[ds[[SIGNAL_COLS[s] for s in SIGNALS]].notna().all(axis=1)]
        if not len(cc):
            continue
        y = cc.correct.to_numpy(int)
        # reliability
        fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
        for ax, s in zip(axes, SIGNALS):
            conf = cc[SIGNAL_COLS[s]].to_numpy(float)
            rb = M.reliability_bins(conf, y, ap["n_bins"])
            e = np.array(rb["edges"])
            mids = (e[:-1] + e[1:]) / 2
            acc = np.array(rb["accuracy"], float)
            cnt = np.array(rb["count"])
            ax.bar(mids[cnt > 0], acc[cnt > 0], width=e[1] - e[0], edgecolor="k", alpha=0.7, label="accuracy")
            ax.plot([0, 1], [0, 1], "k--", lw=1)
            ax.set_title(f"{s} (ECE {M.ece(conf, y, ap['n_bins']):.3f})")
            ax.set_xlabel("confidence")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        axes[0].set_ylabel("accuracy")
        fig.suptitle(f"Reliability — {model} — {setting} (N={len(cc)})")
        fig.tight_layout()
        p = os.path.join(out_dir, f"reliability_{setting}.png")
        fig.savefig(p, dpi=110)
        plt.close(fig)
        paths.append(p)
        # risk-coverage
        fig, ax = plt.subplots(figsize=(6, 4.5))
        for s in SIGNALS:
            conf = cc[SIGNAL_COLS[s]].to_numpy(float)
            cov, risk = M.risk_coverage(conf, y)
            ax.plot(cov, risk, label=f"{s} (AURC {M.aurc(conf, y):.3f})")
        cov, risk = M.risk_coverage(y.astype(float), y)
        ax.plot(cov, risk, "k:", label="oracle")
        ax.axhline(1 - y.mean(), color="gray", ls="--", lw=1, label="random")
        ax.axvline(ap["coverage"], color="gray", lw=0.8)
        ax.set_xlabel("coverage")
        ax.set_ylabel("risk (error rate)")
        ax.set_title(f"Risk–coverage — {model} — {setting}")
        ax.legend(fontsize=8)
        fig.tight_layout()
        p = os.path.join(out_dir, f"risk_coverage_{setting}.png")
        fig.savefig(p, dpi=110)
        plt.close(fig)
        paths.append(p)
        # histograms
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
        bins = M.equal_width_edges(ap["n_bins"])
        for ax, s in zip(axes, SIGNALS):
            conf = cc[SIGNAL_COLS[s]].to_numpy(float)
            ax.hist([conf[y == 1], conf[y == 0]], bins=bins, label=["correct", "wrong"], color=["tab:green", "tab:red"])
            ax.set_title(s)
            ax.set_xlabel("confidence")
        axes[0].legend()
        fig.suptitle(f"Signal histograms — {model} — {setting}")
        fig.tight_layout()
        p = os.path.join(out_dir, f"histograms_{setting}.png")
        fig.savefig(p, dpi=110)
        plt.close(fig)
        paths.append(p)
    return paths


# ------------------------------------------------------------------------------------- entrypoints
def report_from_csv(csv_path: str, out_dir: str | None = None, ap: dict | None = None, figures: bool = True) -> dict:
    out_dir = out_dir or os.path.dirname(os.path.abspath(csv_path))
    os.makedirs(out_dir, exist_ok=True)
    ap = ap or analysis_params(default_config())
    df = read_csv(csv_path)
    summary = analyze_frame(df, ap)
    dump_json(summary, os.path.join(out_dir, "summary.json"))
    with open(os.path.join(out_dir, "results_table.md"), "w", encoding="utf-8", newline="\n") as f:
        f.write(results_markdown(summary))
    if figures:
        make_figures(df, os.path.join(out_dir, "figures"), ap)
    return summary


def _find_csvs(paths) -> list[str]:
    out = []
    for p in paths:
        if os.path.isdir(p):
            out += glob.glob(os.path.join(p, "**", "per_question.csv"), recursive=True)
        elif p.endswith(".zip"):
            out.append(p)
        else:
            out.append(p)
    return sorted(set(out))


def _read_any(p) -> pd.DataFrame:
    if p.endswith(".zip"):
        with zipfile.ZipFile(p) as z:
            name = [n for n in z.namelist() if n.endswith("per_question.csv")][0]
            with z.open(name) as f:
                return read_csv(f)
    return read_csv(p)


def combine(paths, out_dir: str, ap: dict | None = None, prereg_path: str | None = None) -> dict:
    """Recompute every model from its CSV, then Holm across all pairs x non-flagged models per setting."""
    ap = ap or analysis_params(default_config())
    os.makedirs(out_dir, exist_ok=True)
    frames = [_read_any(p) for p in _find_csvs(paths)]
    frames = [f for f in frames if not f.model.iloc[0].endswith("_pilot")]
    by_model = {}
    for f in frames:
        m = f.model.iloc[0]
        if m in by_model:
            raise ValueError(f"model {m} appears twice")
        by_model[m] = (f, analyze_frame(f, ap))
    combined = {"analysis_params": ap, "models": sorted(by_model), "settings": {}}
    for setting in SETTINGS:
        fam, keys = [], []
        for m, (_, s) in sorted(by_model.items()):
            b = s["settings"].get(setting)
            if b is None or b["validity"]["flagged"]:
                continue
            for pk, pr in b["pairs"].items():
                fam.append(pr["delong"]["p"])
                keys.append((m, pk))
        adj = dict(zip(keys, S.holm(fam)))
        per_model = {}
        for m, (_, s) in sorted(by_model.items()):
            b = s["settings"].get(setting)
            if b is None:
                continue
            for pk, pr in b["pairs"].items():
                pr["p_holm_across_models"] = adj.get((m, pk))
                pr["supported_across_models"] = _supported(adj.get((m, pk)), pr["auroc"]["ci"], ap["alpha"]) \
                    if (m, pk) in adj else None
            b["hypotheses_across_models"] = (hypotheses(b, "p_holm_across_models", ap["alpha"])
                                             if not b["validity"]["flagged"] else None)
            per_model[m] = b
        combined["settings"][setting] = {
            "holm_family_size": len(fam), "excluded_flagged_models":
                sorted(m for m, (_, s) in by_model.items() if s["settings"].get(setting, {}).get("validity", {}).get("flagged")),
            "per_model": per_model}
    dump_json(combined, os.path.join(out_dir, "summary_all.json"))
    md = combined_markdown(combined)
    with open(os.path.join(out_dir, "results_all.md"), "w", encoding="utf-8", newline="\n") as f:
        f.write(md)
    figs = forest_plot(combined, os.path.join(out_dir, "figures"))
    bundle = os.path.join(out_dir, "share_bundle_all.zip")
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out_dir, "summary_all.json"), "summary_all.json")
        z.write(os.path.join(out_dir, "results_all.md"), "results_all.md")
        for p in figs:
            z.write(p, "figures/" + os.path.basename(p))
        for m, (f, s) in sorted(by_model.items()):
            slug = m.split("/")[-1]
            z.writestr(f"{slug}/per_question.csv", f.to_csv(index=False, lineterminator="\n"))
            z.writestr(f"{slug}/summary.json", json.dumps(clean_json(s), indent=2, sort_keys=True) + "\n")
        if prereg_path and os.path.exists(prereg_path):
            z.write(prereg_path, "PREREGISTRATION.md")
    combined["bundle"] = bundle
    return combined


def combined_markdown(c: dict) -> str:
    L = ["# Combined results across models", "",
         "Recomputed from each model's `per_question.csv`. Holm correction across all pairs × non-flagged models "
         "within a setting. **The primary family is the CoT setting.**", ""]
    for setting, sb in c["settings"].items():
        L += [f"## Setting: {setting}{' (primary)' if setting == 'cot' else ' (secondary)'}", "",
              f"Holm family size: {sb['holm_family_size']}; flagged models excluded: {sb['excluded_flagged_models'] or 'none'}", "",
              "| model | N | accuracy | AUROC stated | AUROC prob | AUROC agree | ECE stated | ECE prob | ECE agree |",
              "|---|---|---|---|---|---|---|---|---|"]
        for m, b in sb["per_model"].items():
            sg = b["signals"]
            L.append(f"| {m}{' (FLAGGED)' if b['validity']['flagged'] else ''} | {b['n']} | {_f(b['accuracy']['value'])} | "
                     + " | ".join(_fci(sg[s]["auroc"]) for s in SIGNALS) + " | "
                     + " | ".join(_fci(sg[s]["ece"]) for s in SIGNALS) + " |")
        L += ["", "| model | pair | ΔAUROC [95% CI] | p | p (Holm, across models) | supported |", "|---|---|---|---|---|---|"]
        for m, b in sb["per_model"].items():
            for pk, pr in b["pairs"].items():
                sup = pr.get("supported_across_models")
                L.append(f"| {m} | {pk} | {_fci(pr['auroc'])} | {_p(pr['delong']['p'])} | {_p(pr.get('p_holm_across_models'))} | "
                         f"{'excluded' if sup is None else ('yes' if sup else 'no')} |")
        L += ["", "| model | H1 | H2 | H3 |", "|---|---|---|---|"]
        for m, b in sb["per_model"].items():
            h = b.get("hypotheses_across_models")
            if h is None:
                L.append(f"| {m} | excluded (flagged) | | |")
            else:
                L.append(f"| {m} | {h['H1']['supported']} | {h['H2']['supported']} | {h['H3']['supported']} ({h['H3']['direction']}) |")
        L.append("")
    return "\n".join(L) + "\n"


def forest_plot(c: dict, out_dir: str) -> list[str]:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for setting, sb in c["settings"].items():
        models = list(sb["per_model"])
        if not models:
            continue
        fig, ax = plt.subplots(figsize=(7, 1.2 + 0.9 * len(models)))
        for j, s in enumerate(SIGNALS):
            for i, m in enumerate(models):
                a = sb["per_model"][m]["signals"][s]["auroc"]
                yv = i + (j - 1) * 0.22
                lo, hi = a["ci"]
                if lo is not None:
                    ax.plot([lo, hi], [yv, yv], color=f"C{j}")
                ax.plot(a["value"], yv, "o", color=f"C{j}", label=s if i == 0 else None)
        ax.set_yticks(range(len(models)))
        ax.set_yticklabels([m.split("/")[-1] for m in models])
        ax.set_xlabel("AUROC (95% bootstrap CI)")
        ax.set_title(f"AUROC by model — {setting}")
        ax.legend(fontsize=8)
        fig.tight_layout()
        p = os.path.join(out_dir, f"auroc_forest_{setting}.png")
        fig.savefig(p, dpi=110)
        plt.close(fig)
        paths.append(p)
    return paths


def main(argv=None):
    ap_ = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap_.add_argument("--from-csv", help="per_question.csv to analyse")
    ap_.add_argument("--combine", nargs="+", help="per-model output folders / CSVs / share bundles")
    ap_.add_argument("--out", help="output directory (default: the CSV's folder)")
    ap_.add_argument("--no-figures", action="store_true")
    ap_.add_argument("--prereg", default=None)
    a = ap_.parse_args(argv)
    if a.from_csv:
        report_from_csv(a.from_csv, a.out, figures=not a.no_figures)
        print(f"wrote summary.json to {a.out or os.path.dirname(os.path.abspath(a.from_csv))}")
    elif a.combine:
        c = combine(a.combine, a.out or "combined", prereg_path=a.prereg)
        print(f"wrote {c['bundle']}")
    else:
        ap_.error("give --from-csv or --combine")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile medqa_src/medqa_conf/sanity.py
"""sanity_report.md (PREREGISTRATION.md section 9)."""
from __future__ import annotations

import collections

import numpy as np

from . import metrics as M
from . import parsing

SIGNAL_COLS = ("conf_stated", "conf_prob", "conf_agree")


def _pct(x):
    return f"{100 * x:.1f}%"


def _dist(series):
    c = collections.Counter(series)
    n = sum(c.values())
    return ", ".join(f"{k}: {v} ({_pct(v / n)})" for k, v in sorted(c.items()))


def sanity_stats(records, df, data_info, determinism, validity_by_setting) -> dict:
    cot = df[df.setting == "cot"]
    st = {}
    st["greedy_answer_parse_fail"] = float((cot.answer_parse_method == "none").mean())
    st["greedy_answer_parse_methods"] = dict(collections.Counter(cot.answer_parse_method))
    st["greedy_conf_missing_before_recovery"] = float(cot.stated_missing_before_recovery.mean())
    st["greedy_conf_missing_after_recovery"] = float(cot.conf_stated.isna().mean())
    st["n_stated_recovered"] = int(cot.stated_recovered.sum())
    n_samp = sum(len(r["cot"]["samples"]) for r in records)
    n_unparsed = sum(parsing.parse_answer(s["text"])[0] is None for r in records for s in r["cot"]["samples"])
    st["sample_answer_parse_fail"] = n_unparsed / n_samp if n_samp else float("nan")
    st["greedy_truncation"] = float(cot.greedy_truncated.mean())
    st["sample_truncation"] = float(cot.n_samples_truncated.sum() / n_samp) if n_samp else float("nan")
    st["cot_forced_answer"] = float(cot.forced_answer.mean())
    mm = cot.parsed_vs_argmax_mismatch.dropna()
    st["cot_parsed_vs_argmax_mismatch"] = float(mm.mean()) if len(mm) else float("nan")
    st["cot_parsed_vs_argmax_n"] = int(len(mm))
    for setting in ("cot", "direct"):
        ds = df[df.setting == setting]
        q = np.percentile(ds.letter_mass, [0, 5, 25, 50, 75, 95, 100]).tolist()
        st[f"{setting}_letter_mass_quantiles"] = dict(zip(["min", "p5", "p25", "median", "p75", "p95", "max"], q))
        st[f"{setting}_any_letter_missing_topk"] = float((ds.n_letters_missing_topk > 0).mean())
        st[f"{setting}_zero_letter_mass"] = int((ds.letter_mass == 0).sum())
        st[f"{setting}_conf_missing"] = int(ds.conf_stated.isna().sum())
        st[f"{setting}_stated_fraction_flag"] = int(ds.stated_fraction_flag.sum())
        st[f"{setting}_chosen_letters"] = dict(collections.Counter(ds.answer))
        st[f"{setting}_accuracy"] = float(ds.correct.mean())
        sig = {}
        for c in SIGNAL_COLS:
            v = ds[c].dropna().to_numpy(float)
            if not len(v):
                sig[c] = None
                continue
            uniq, cnt = np.unique(v, return_counts=True)
            idx = M.bin_index(v, M.equal_width_edges(15))
            sig[c] = {"hist_15bins": np.bincount(idx, minlength=15).tolist(),
                      "frac_tied": float(cnt[cnt > 1].sum() / len(v)), "n_unique": int(len(uniq))}
            if c == "conf_stated":
                top = sorted(zip(cnt, uniq), key=lambda t: (-t[0], t[1]))[:5]
                sig[c]["top5_values"] = [[float(u), int(k)] for k, u in top]
        st[f"{setting}_signals"] = sig
    st["gold_letters"] = dict(collections.Counter(cot.gold))
    st["determinism"] = determinism
    st["validity"] = validity_by_setting
    st["data"] = data_info
    return st


def sanity_markdown(st: dict, model: str) -> str:
    d = st["data"]
    L = [f"# Sanity report: {model}", "",
         "## [fail] checks", "",
         f"- PASS dataset counts and schema: test split = {d['test']['n_questions']} rows, 4 options A–D, gold in A–D "
         f"(run would have stopped otherwise). Split used: {d['used_split']} ({d['used']['n_questions']} questions).",
         f"- PASS no NaN/inf in any returned logprob (run would have stopped otherwise).", "",
         "## Data", "",
         f"- Options cleaned (whitespace / stray trailing quote): test {d['test']['n_options_cleaned']}, "
         f"used split {d['used']['n_options_cleaned']}",
         f"- Gold answer text ≠ option text after cleaning: {d['used']['n_answer_text_mismatch']}",
         f"- Gold-letter distribution (used split): {d['used']['gold_distribution']}",
         f"- Gold-letter distribution (test split): {d['test']['gold_distribution']}",
         f"- meta_info (used split): {d['used']['meta_info_distribution']}", "",
         "## Parsing", "",
         f"- Greedy answer parse failure: {_pct(st['greedy_answer_parse_fail'])} (methods: {st['greedy_answer_parse_methods']})",
         f"- Greedy confidence missing before recovery: {_pct(st['greedy_conf_missing_before_recovery'])}; "
         f"after recovery: {_pct(st['greedy_conf_missing_after_recovery'])} ({st['n_stated_recovered']} recovered)",
         f"- Sample answer parse failure: {_pct(st['sample_answer_parse_fail'])}",
         f"- CoT forced_answer (no 'Answer:' in G): {_pct(st['cot_forced_answer'])}",
         f"- CoT parsed letter ≠ argmax p: {_pct(st['cot_parsed_vs_argmax_mismatch'])} (of {st['cot_parsed_vs_argmax_n']} parsed)",
         f"- Direct confidence missing: {st['direct_conf_missing']}", "",
         "## Letter distribution", ""]
    for s in ("cot", "direct"):
        q = st[f"{s}_letter_mass_quantiles"]
        L.append(f"- {s}: letter_mass " + ", ".join(f"{k} {v:.3f}" for k, v in q.items())
                 + f"; any letter missing from top-k: {_pct(st[f'{s}_any_letter_missing_topk'])}; "
                 f"zero letter mass: {st[f'{s}_zero_letter_mass']}")
    L += ["", "## Truncation (finish_reason == length)", "",
          f"- Greedy: {_pct(st['greedy_truncation'])}; samples: {_pct(st['sample_truncation'])}", "",
          "## Position bias (chosen vs gold letters)", "",
          f"- gold: {st['gold_letters']}"]
    for s in ("cot", "direct"):
        L.append(f"- {s} chosen: {st[f'{s}_chosen_letters']} (accuracy {st[f'{s}_accuracy']:.3f})")
    L += ["", "## Signals (15 equal-width bins on [0,1])", ""]
    for s in ("cot", "direct"):
        for c, v in st[f"{s}_signals"].items():
            if v is None:
                L.append(f"- {s} {c}: no values")
                continue
            extra = f"; top-5 values {v['top5_values']}" if "top5_values" in v else ""
            L.append(f"- {s} {c}: hist {v['hist_15bins']}; fraction tied {v['frac_tied']:.3f}; "
                     f"unique {v['n_unique']}{extra}; fraction-format flags {st[f'{s}_stated_fraction_flag'] if c == 'conf_stated' else '-'}")
    det = st["determinism"]
    L += ["", "## Greedy determinism", "",
          f"- Re-ran CoT greedy on {det['n']} questions: {det['identical']}/{det['n']} byte-identical"
          + (f" (differing qids: {det['qids_differing']})" if det["qids_differing"] else ""), "",
          "## Validity flags (PREREGISTRATION.md section 8)", ""]
    for s, v in st["validity"].items():
        L.append(f"- {s}: {'FLAGGED' if v['flagged'] else 'not flagged'} — values "
                 + ", ".join(f"{k} {x:.3f}" for k, x in v["values"].items())
                 + " — flags " + ", ".join(f"{k}={x}" for k, x in v["flags"].items()))
    return "\n".join(L) + "\n"


In [ ]:
%%writefile medqa_src/medqa_conf/bundle.py
"""run_log.json, raw_examples.md and the share bundle."""
from __future__ import annotations

import importlib
import json
import os
import platform
import sys
import zipfile

from . import prompts

MAX_BUNDLE_BYTES = 5 * 1024 * 1024
BUNDLE_FILES = ("per_question.csv", "summary.json", "results_table.md", "sanity_report.md", "run_log.json",
                "raw_examples.md")


def package_versions() -> dict:
    out = {"python": sys.version.split()[0], "platform": platform.platform()}
    for name in ("numpy", "scipy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "vllm",
                 "datasets", "huggingface_hub", "tokenizers"):
        try:
            mod = sys.modules.get(name) or importlib.import_module(name)
            out[name] = getattr(mod, "__version__", "?")
        except Exception:
            out[name] = None
    try:
        import torch
        out["cuda"] = torch.version.cuda
        out["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:
        out.setdefault("gpus", [])
    return out


def _trunc(text: str, limit: int = 1500) -> str:
    if text is None:
        return ""
    if len(text) <= limit:
        return text
    h = limit // 2
    return text[:h] + f"\n…[{len(text) - limit} characters truncated]…\n" + text[-h:]


def raw_examples_markdown(df, records, questions, model: str) -> str:
    cot = df[df.setting == "cot"].copy()
    rec = {r["qid"]: r for r in records}
    qs = {q.qid: q for q in questions}
    cc = cot.sort_values(["conf_prob", "qid"], ascending=[False, True], kind="stable")
    picks = [("confident & correct (top conf_prob)", cc[cc.correct == 1].head(4)),
             ("confident & wrong (top conf_prob)", cc[cc.correct == 0].head(4))]
    prob = []
    for cond in (cot.answer_parse_method == "none", cot.forced_answer == 1, cot.stated_recovered == 1,
                 cot.conf_stated.isna()):
        for qid in cot[cond].qid:
            if qid not in prob:
                prob.append(qid)
    picks.append(("parse failure / forced answer / recovered confidence",
                  cot[cot.qid.isin(prob[:4])].set_index("qid").loc[prob[:4]].reset_index()))
    L = [f"# Raw examples: {model} (CoT setting)", "",
         "Reasoning longer than 1,500 characters is truncated in the middle.", ""]
    for title, rows in picks:
        L += [f"## {title}", ""]
        if not len(rows):
            L += ["_none_", ""]
        for _, row in rows.iterrows():
            r, q = rec[int(row.qid)], qs[int(row.qid)]
            L += [f"### qid {int(row.qid)} — gold {row.gold}, official answer {row.answer}, "
                  f"written {row.answer_parsed or 'none'} ({row.answer_parse_method})", "",
                  f"conf_stated {row.conf_stated}, conf_prob {row.conf_prob:.3f}, conf_agree {row.conf_agree}, "
                  f"samples `{row.sample_answers}`, forced_answer {int(row.forced_answer)}, "
                  f"stated_recovered {int(row.stated_recovered)}, letter_mass {row.letter_mass:.3f}", "",
                  "**Question block**", "", "```", prompts.question_block(q), "```", "",
                  "**Greedy output G**", "", "```", _trunc(r["cot"]["greedy"]["text"]), "```", ""]
            if r["cot"].get("recovery"):
                L += [f"Recovery generation: `{r['cot']['recovery']['text']!r}`", ""]
    return "\n".join(L) + "\n"


def write_run_log(path: str, payload: dict) -> None:
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        json.dump(payload, f, indent=2, sort_keys=True, default=str)
        f.write("\n")


def make_share_bundle(out_dir: str, slug: str, prereg_path: str | None, code_dir: str | None = None) -> str:
    path = os.path.join(out_dir, f"share_bundle_{slug}.zip")
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in BUNDLE_FILES:
            p = os.path.join(out_dir, f)
            if os.path.exists(p):
                z.write(p, f)
        figdir = os.path.join(out_dir, "figures")
        if os.path.isdir(figdir):
            for f in sorted(os.listdir(figdir)):
                z.write(os.path.join(figdir, f), "figures/" + f)
        if prereg_path and os.path.exists(prereg_path):
            z.write(prereg_path, "PREREGISTRATION.md")
        if code_dir and os.path.isdir(code_dir):
            for root, _, files in os.walk(code_dir):
                for f in sorted(files):
                    if f.endswith(".py"):
                        full = os.path.join(root, f)
                        z.write(full, "code/" + os.path.relpath(full, code_dir).replace(os.sep, "/"))
    size = os.path.getsize(path)
    if size >= MAX_BUNDLE_BYTES:
        raise RuntimeError(f"share bundle is {size / 1e6:.2f} MB (limit 5 MB)")
    return path


In [ ]:
%%writefile medqa_src/medqa_conf/pipeline.py
"""End-to-end orchestration: generation -> per_question.csv -> report -> sanity -> bundle.

CPU mock run:  python -m medqa_conf.pipeline --mock --n 30 --out outputs_mock
"""
from __future__ import annotations

import argparse
import dataclasses
import json
import os
import time

from . import bundle, generate, prompts, report, sanity, signals

TEST_N = 1273
ANALYSIS_FULL_ESTIMATE_S = 900  # conservative allowance for bootstrap/report on 1,273 questions


class PilotAbort(RuntimeError):
    pass


def _garbage_stats(records):
    texts = [r["cot"]["greedy"]["text"] for r in records]
    printable = [sum(ch.isprintable() or ch in "\n\t" for ch in t) / max(1, len(t)) for t in texts]
    rep = []
    for t in texts:
        lines = [l for l in t.splitlines() if l.strip()]
        rep.append(max([lines.count(l) for l in set(lines)] or [0]))
    return {"min_printable_ratio": min(printable) if printable else None,
            "max_repeated_line": max(rep) if rep else None,
            "empty_outputs": sum(not t.strip() for t in texts)}


def pilot_report(records, df, questions, backend, cfg, gen_status, timings) -> tuple[str, dict]:
    rec = {r["qid"]: r for r in records}
    L = [f"# PILOT report: {cfg.model_id}", "",
         f"Pilot uses {len(questions)} **train** questions (indices {[q.qid for q in questions]}). "
         "Not part of the study.", ""]
    for q in questions[:3]:
        r = rec[q.qid]
        cp = backend.render(prompts.cot_message(q))
        dp = backend.render(prompts.direct_message(q))
        dist = generate.letter_distribution({int(k): v for k, v in r["direct"]["logprobs"].items()},
                                            backend.letter_variants)
        L += [f"## qid {q.qid} (gold {q.gold})", "", "### Rendered CoT prompt", "", "```", cp, "```", "",
              "### Raw greedy CoT output G", "", "```", r["cot"]["greedy"]["text"], "```", "",
              f"finish={r['cot']['greedy']['finish']}, tokens={r['cot']['greedy']['tokens']}, "
              f"sample answers={[s['text'][-60:] for s in r['cot']['samples']]}", "",
              "### Rendered Direct prompt + prefill", "", "```", dp + prompts.DIRECT_LETTER_PREFILL, "```", "",
              f"Direct p = { {k: round(v, 4) for k, v in dist['p'].items()} }, letter_mass {dist['letter_mass']:.4f}; "
              f"confidence generation after `Answer: {dist['argmax']}\\nConfidence:` = "
              f"{r['direct']['confidence_gen']['text']!r}", ""]
    cot = df[df.setting == "cot"]
    dirr = df[df.setting == "direct"]
    n_tok = sum(r["cot"]["greedy"]["tokens"] + sum(s["tokens"] for s in r["cot"]["samples"]) for r in records)
    gs = gen_status["generation_seconds"]
    spq = gen_status["seconds_per_question"] or 0.0
    setup = sum(timings.get(k, 0.0) for k in ("model_download", "smoke_test", "model_load", "install"))
    proj_s = setup + spq * TEST_N + timings.get("determinism", 0.0) + max(timings.get("analysis", 0.0), ANALYSIS_FULL_ESTIMATE_S)
    proj = {"seconds_per_question": spq, "generated_tokens": n_tok,
            "throughput_tokens_per_s": n_tok / gs if gs else None,
            "setup_seconds": setup, "projected_full_hours": proj_s / 3600,
            "max_hours": cfg.max_projected_hours, "abort": proj_s / 3600 > cfg.max_projected_hours}
    g = _garbage_stats(records)
    warn = []
    if (cot.answer_parse_method == "none").mean() > 0.10:
        warn.append("greedy answer parse failure > 10%")
    for s, d in (("cot", cot), ("direct", dirr)):
        if d.letter_mass.median() < 0.5:
            warn.append(f"{s} median letter_mass < 0.5")
    if g["min_printable_ratio"] is not None and g["min_printable_ratio"] < 0.95:
        warn.append("non-printable characters in greedy output (possible fp16 garbage)")
    if g["max_repeated_line"] and g["max_repeated_line"] >= 10:
        warn.append("a greedy output repeats one line >= 10 times (degenerate output?)")
    if g["empty_outputs"]:
        warn.append(f"{g['empty_outputs']} empty greedy outputs")
    L += ["## Rates", "",
          f"- Greedy answer parse rate: {1 - (cot.answer_parse_method == 'none').mean():.1%} "
          f"(methods {cot.answer_parse_method.value_counts().to_dict()})",
          f"- Greedy confidence parse rate before recovery: {1 - cot.stated_missing_before_recovery.mean():.1%}; "
          f"after: {1 - cot.conf_stated.isna().mean():.1%}",
          f"- Direct confidence parse rate: {1 - dirr.conf_stated.isna().mean():.1%}",
          f"- forced_answer: {cot.forced_answer.mean():.1%}; parsed-vs-argmax mismatch: "
          f"{cot.parsed_vs_argmax_mismatch.mean():.1%}",
          f"- letter_mass median: CoT {cot.letter_mass.median():.3f} (min {cot.letter_mass.min():.3f}), "
          f"Direct {dirr.letter_mass.median():.3f} (min {dirr.letter_mass.min():.3f})",
          f"- Truncation: greedy {cot.greedy_truncated.mean():.1%}, samples {cot.n_samples_truncated.sum() / (5 * len(cot)):.1%}",
          f"- Accuracy (train, 20 q — noisy): CoT {cot.correct.mean():.2f}, Direct {dirr.correct.mean():.2f}",
          f"- Output health: {g}", "",
          "## Throughput and projection", "",
          f"- {spq:.1f} s per question (all 6 calls), {proj['throughput_tokens_per_s'] or 0:.0f} generated tokens/s",
          f"- Setup (install + download + smoke test + load): {setup / 60:.1f} min",
          f"- Projected full run (1,273 questions): **{proj['projected_full_hours']:.2f} h** "
          f"(limit {cfg.max_projected_hours} h) → {'ABORT' if proj['abort'] else 'OK'}",
          "- The projection is conservative: the full run batches 50 questions per chunk vs 20 here.", "",
          "## Warnings", ""] + ([f"- {w}" for w in warn] or ["- none"])
    proj["warnings"] = warn
    proj["output_health"] = g
    return "\n".join(L) + "\n", proj


def run(cfg, backend, questions, data_info, out_dir, *, model_revision=None, dataset_revision=None,
        resume_roots=(), time_budget_s=None, t_start=None, timings=None, env=None, prereg_path=None,
        code_dir=None, backend_note=None, log=print) -> dict:
    t_start = t_start if t_start is not None else time.time()
    timings = dict(timings or {})
    os.makedirs(out_dir, exist_ok=True)
    run_log = {"config": cfg.to_dict(), "config_hash": cfg.config_hash(), "model_id": cfg.model_id,
               "model_revision": model_revision, "dataset_id": cfg.dataset_id, "dataset_revision": dataset_revision,
               "seeds": {"base_seed": cfg.base_seed, "sample_seed": "base_seed + qid", "boot_seed": cfg.boot_seed,
                         "cv_seed": cfg.cv_seed},
               "pilot": cfg.pilot, "backend": backend.info(), "backend_note": backend_note,
               "environment": env or {}, "data": data_info, "timings_seconds": timings}
    if questions:
        run_log["example_rendered_cot_prompt"] = backend.render(prompts.cot_message(questions[0]))

    t0 = time.time()
    st = generate.run_generation(backend, cfg, questions, out_dir, model_revision=model_revision,
                                 resume_roots=resume_roots, time_budget_s=time_budget_s, t_start=t_start, log=log)
    timings["generation"] = time.time() - t0
    run_log["generation"] = {k: v for k, v in st.items() if k != "records"}
    records = st["records"]
    if not st["complete"]:
        run_log["status"] = "incomplete"
        run_log["versions"] = bundle.package_versions()
        bundle.write_run_log(os.path.join(out_dir, "run_log.json"), run_log)
        log(f"[pipeline] INCOMPLETE: {len(records)}/{len(questions)} questions. Attach this notebook's output "
            "as input and re-run to resume.")
        return {"status": "incomplete", "out_dir": out_dir, "n_done": len(records)}

    t0 = time.time()
    det = generate.determinism_check(backend, cfg, questions, records, cfg.determinism_n)
    timings["determinism"] = time.time() - t0

    t0 = time.time()
    df = signals.records_to_frame(records, cfg.model_id + ("_pilot" if cfg.pilot else ""), backend.letter_variants)
    csv_path = os.path.join(out_dir, "per_question.csv")
    df.to_csv(csv_path, index=False, lineterminator="\n")
    ap = report.analysis_params(cfg)
    summary = report.report_from_csv(csv_path, out_dir, ap)  # computed from the saved CSV only
    df = report.read_csv(csv_path)
    timings["analysis"] = time.time() - t0

    t0 = time.time()
    validity = {s: b["validity"] for s, b in summary["settings"].items()}
    sst = sanity.sanity_stats(records, df, data_info, det, validity)
    with open(os.path.join(out_dir, "sanity_report.md"), "w", encoding="utf-8", newline="\n") as f:
        f.write(sanity.sanity_markdown(sst, cfg.model_id))
    generate.write_raw_generations(records, os.path.join(out_dir, "raw_generations.jsonl.gz"))
    with open(os.path.join(out_dir, "raw_examples.md"), "w", encoding="utf-8", newline="\n") as f:
        f.write(bundle.raw_examples_markdown(df, records, questions, cfg.model_id))
    result = {"status": "complete", "out_dir": out_dir, "csv": csv_path, "summary": summary}
    if cfg.pilot:
        md, proj = pilot_report(records, df, questions, backend, cfg, st, timings)
        with open(os.path.join(out_dir, "pilot_report.md"), "w", encoding="utf-8", newline="\n") as f:
            f.write(md)
        run_log["pilot_projection"] = proj
        result["pilot_report"] = md
        result["projection"] = proj
    timings["reports"] = time.time() - t0
    timings["total_wall"] = time.time() - t_start
    run_log["determinism"] = det
    run_log["status"] = "complete"
    run_log["versions"] = bundle.package_versions()
    bundle.write_run_log(os.path.join(out_dir, "run_log.json"), run_log)
    result["bundle"] = bundle.make_share_bundle(out_dir, cfg.model_slug, prereg_path, code_dir)
    log(f"[pipeline] done: {result['bundle']} ({os.path.getsize(result['bundle']) / 1e6:.2f} MB)")
    return result


def check_pilot(result) -> None:
    """Raise after all pilot files are written if the projected runtime is too long."""
    p = result.get("projection")
    if p and p["abort"]:
        raise PilotAbort(f"projected full runtime {p['projected_full_hours']:.2f} h exceeds {p['max_hours']} h")


# ------------------------------------------------------------------------------------ mock CLI
def run_mock(n: int, out_root: str, pilot: bool = False, accuracy: float = 0.62, model_id="mock/Mock-7B",
             time_budget_s=None, resume_roots=(), prereg_path=None, code_dir=None, log=print):
    from config import Config

    from . import data
    from .backends import MockBackend

    cfg = dataclasses.replace(Config(), model_id=model_id, pilot=pilot)
    test_rows = data.synthetic_rows(n, seed=1)
    train_rows = data.synthetic_rows(200, seed=2)
    qs, info = data.load_questions(cfg, train_rows, test_rows, expected_test=n, expected_train=None)
    key = {q.question: q.gold for q in qs}
    backend = MockBackend(key, accuracy=accuracy, seed=0)
    out_dir = os.path.join(out_root, cfg.model_slug)
    res = run(cfg, backend, qs, info, out_dir, model_revision="mock", dataset_revision="synthetic",
              resume_roots=resume_roots, time_budget_s=time_budget_s, prereg_path=prereg_path,
              code_dir=code_dir, env={"note": "CPU mock run"}, log=log)
    return cfg, qs, res


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--mock", action="store_true", required=True)
    a.add_argument("--n", type=int, default=30)
    a.add_argument("--out", default="outputs_mock")
    a.add_argument("--pilot", action="store_true")
    a.add_argument("--prereg", default="PREREGISTRATION.md")
    a = a.parse_args(argv)
    t = time.time()
    _, _, res = run_mock(a.n, a.out, pilot=a.pilot, prereg_path=a.prereg, code_dir="src")
    print(json.dumps({"status": res["status"], "bundle": res.get("bundle"), "seconds": round(time.time() - t, 1)}))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile medqa_src/PREREGISTRATION.md
# Pre-registration: Which confidence signal works for medical QA?

- Registered: 2026-10-10 (git commit date of this file is the authoritative timestamp)
- Status: written and committed **before any GPU run**. No model output on the MedQA test split has been seen.
- Any change to a pre-registered choice after this commit will be listed in a "Deviations" section appended to this file, with date and reason.

## 1. Research question

Which signal best separates right from wrong answers of open LLMs on MedQA (USMLE, 4 options, 1,273 test questions)?

- (a) **stated confidence** (`conf_stated`): the model writes a number from 0 to 100.
- (b) **option probability** (`conf_prob`): the probability of the chosen letter.
- (c) **sample agreement** (`conf_agree`): the share of 5 sampled answers that match the official answer.

Primary measure: AUROC for correctness. Secondary: calibration error (ECE), and selective accuracy when the model answers only its 80% most-confident questions.

## 2. Data

- HF dataset `GBaker/MedQA-USMLE-4-options` (CC-BY-4.0). Splits: train (10,178 rows), test (1,273 rows). The dataset revision (commit hash) is resolved at run time and logged.
- Fields used: `question`, `options` (dict A–D), `answer_idx` (gold letter), `answer` (gold text), `meta_info` (`step1` / `step2&3`).
- The run **fails** unless the test split has exactly 1,273 rows, every row has exactly the 4 option keys A, B, C, D, and every gold letter is in A–D.
- `qid` = row index in the test split. The sha1 of the raw question text is stored.
- Cleaning (only): question text and option texts are stripped of surrounding whitespace; an option text ending in a `"` with an odd number of `"` characters (an unbalanced, stray quote) has that trailing `"` removed and is stripped again. The number of cleaned options is logged. The gold-letter distribution is logged.
- The **test split is used only for the main run.** The pilot uses 20 questions from the **train** split (row indices drawn with `numpy.random.default_rng(BASE_SEED).choice(10178, 20, replace=False)`, sorted).

## 3. Models

One model per run:

1. `Qwen/Qwen2.5-7B-Instruct`
2. `meta-llama/Llama-3.1-8B-Instruct`
3. `mistralai/Mistral-7B-Instruct-v0.3`

- float16 (T4 has no bf16). No model families known to overflow in fp16.
- No system message: everything is in the single user message. (A chat template that injects its own default system text, e.g. Qwen2.5, is used as-is; the rendered prompt is logged.)
- Any "thinking mode" is disabled (`enable_thinking=False` is passed to the chat template).
- The exact HF revision (commit hash) is resolved, downloaded and logged.
- Hardware: Kaggle, 2× T4 16 GB. Backend: vLLM (tensor_parallel_size 2, float16, max_model_len 4096, top-20 logprobs) preferred; transformers fallback if vLLM cannot be installed/started (the switch is logged).

## 4. Prompts (verbatim)

Every prompt is built with the tokenizer's chat template (`add_generation_prompt=True`). "Prefill" means text appended after the generation prompt, as the start of the assistant turn. Prompts are tokenized without adding extra special tokens (the template already contains them).

Shared question block:

```
The following is a multiple-choice question from the US Medical Licensing Examination (USMLE). Choose the single best answer.

Question: {question}
A. {A}
B. {B}
C. {C}
D. {D}
```

**CoT user message** (main, pre-registered setting) = question block followed by:

```

Think through the problem step by step. Then finish with exactly these two lines:
Answer: <A, B, C or D>
Confidence: <a number from 0 to 100: the probability that your answer is correct>
```

**Direct user message** (secondary setting) = question block followed by:

```

Reply with exactly these two lines and nothing else:
Answer: <A, B, C or D>
Confidence: <a number from 0 to 100: the probability that your answer is correct>
```

## 5. Model calls per question

| # | Call | Settings |
|---|---|---|
| 1 | CoT greedy → text `G` | temperature 0, max_new_tokens 1024 |
| 2 | CoT samples → `S1..S5` | n = 5, temperature 0.7, top_p 1.0, no top-k, no repetition penalty, max_new_tokens 1024, seed = BASE_SEED + qid |
| 3 | CoT letter probability | prompt = chat prompt + `G` up to and including its last `Answer:` (exact, case-sensitive string). If `G` has no `Answer:`, prefix = `G.rstrip() + "\nAnswer:"` and flag `forced_answer`. Read next-token log-probabilities → letter distribution `p`. **Official CoT answer X = argmax p.** |
| 4 | CoT stated confidence | parsed from `G`. Only if none can be parsed: prefill = CoT prefix + `" {X}\nConfidence:"`, greedy, max 6 new tokens; flag `stated_recovered`. |
| 5 | Direct letter probability | prefill `Answer:`; next-token log-probabilities → `p`. **Official Direct answer X = argmax p.** |
| 6 | Direct stated confidence | prefill `Answer: {X}\nConfidence:`, greedy, max 6 new tokens. |

Seeds: `BASE_SEED = 12345`; engine seed = BASE_SEED; bootstrap seed = 2026; cross-fitting seed = 2026. Model-provided generation-config defaults (e.g. top_k, repetition_penalty) are overridden by the values above; EOS token ids are taken from the model's generation config.

### Letter distribution

- For each letter L, sum the probabilities of the single-token variants `"L"` and `" L"` (the variants present in the vocabulary are detected and their token ids logged).
- `letter_mass` = total probability on the 4 letters before renormalising; `p` = distribution renormalised over the 4 letters.
- With vLLM top-20 logprobs, a variant missing from the top 20 contributes 0; the frequency of any letter missing is logged. With the transformers backend the full distribution is used.
- argmax ties (practically impossible) are broken alphabetically.
- If `letter_mass` is 0 (no letter in the top 20), `p` is uniform and the answer is A; the count is reported.
- In CoT, the letter written in `G` is also parsed; the rate at which it disagrees with argmax p is reported.

### Answer parsing

- Primary: the **last** match of `answer` (case-insensitive word) optionally followed by `**`, then `:`, then optional `**`, optional `(` or `[`, then one capital letter A–D not followed by another letter. Tolerates `**Answer:** C`, `**Answer: C**`, `Answer: (C)`, `Final Answer: C`.
- Fallback: last match of `answer is` followed by an optional `:`/`(`/`**`/"option" and a capital A–D (not followed by a letter).
- Otherwise None. The method used (`answer_line`, `answer_is`, `none`) is recorded.

## 6. Signals (per setting)

- **conf_stated** = parsed number / 100, clipped to [0, 1]. Last match of `confidence` (case-insensitive) with optional `**`, `:`, optional `**`, then a number (`85`, `85.5`, `.9`); `%` and extra spaces tolerated. If the value is ≤ 1 **and** contains a decimal point it is a fraction (used as-is) and `stated_fraction_flag` is set. For the 6-token generations (CoT recovery, Direct), the first number in the generated text is parsed with the same rule. Missing after recovery → NaN; the count is reported.
- **conf_prob** = p[official answer], range [0.25, 1].
- **conf_agree** = (# of the 5 CoT samples whose parsed answer equals the official answer) / 5. An unparsable sample counts as disagreeing. The Direct setting uses the **same 5 CoT samples**.
- All three signals are scored against the **same official answer** X (section 5). y = 1 iff X equals the gold letter.

**Exploratory analyses** (labelled exploratory, excluded from main conclusions): majority-vote accuracy over the 5 samples (ties broken by the setting's official answer, then alphabetically; with no parsable sample, the official answer); entropy (natural log) of the 5 sample answers, with "unparsed" as its own category; rank-average of conf_prob and conf_agree (mean of average ranks); results by `meta_info`; ECE after 5-fold cross-fitted Platt scaling (unregularised logistic regression on the raw signal, folds shuffled with seed 2026).

## 7. Metrics and statistics

Per model × setting × signal:

- **Accuracy** with Wilson 95% CI.
- **AUROC** for correctness, ties count ½ (identical to `sklearn.metrics.roc_auc_score`).
- **ECE**: 15 equal-width bins on [0, 1], edges `i/15`; bins left-closed, the last bin also includes 1.0; weighted mean |accuracy − mean confidence|. (The function accepts custom edges.)
- **Adaptive ECE**: 15 equal-mass bins: items sorted by confidence (stable sort, ties kept in qid order) and split into 15 contiguous groups with `numpy.array_split`.
- **Brier score**; **overconfidence gap** = mean confidence − accuracy.
- **Selective accuracy at 80% coverage**: k = floor(0.8·N) (1,018 for N = 1,273), questions ranked by confidence, highest first. Ties at the cutoff: exact expected accuracy under random tie-breaking = (correct above the tied group + needed × accuracy of the tied group) / k. References: overall accuracy (random ranking) and oracle min(#correct, k)/k.
- **Risk–coverage curve and AURC**, tie-aware: for every k = 1..N the expected risk under random tie-breaking; AURC = mean over k.
- **Paired bootstrap**: 2,000 resamples of questions; within a model × setting all signals use the same indices; percentile 95% CIs for every metric and every pairwise difference.
- **DeLong**: paired fast DeLong (Sun & Xu 2014) for the 3 pairwise AUROC differences; two-sided p.
- **Holm**: per-model report: across that model's 3 pairs; combined report: across all pairs × (non-flagged) models within a setting. **The primary family is the CoT setting.**
- **Missing values**: primary analysis is complete-case (a question with any NaN signal is dropped for all signals; N used is reported). Sensitivity analysis: missing conf_stated imputed with its median.

## 8. Hypotheses and decision rule

- **H1**: in the CoT setting, conf_stated has a lower AUROC than both conf_prob and conf_agree, for each model.
- **H2**: conf_stated is overconfident (mean confidence − accuracy > 0) and has the highest ECE.
- **H3** (two-sided, no direction predicted): conf_prob and conf_agree differ in AUROC.

**Decision rule.** An AUROC difference is "supported" when the Holm-adjusted DeLong p < 0.05 **and** the paired bootstrap 95% CI of the difference excludes 0 (and, for H1, the difference has the predicted sign). DeLong applies only to AUROC, so H2 is judged by bootstrap alone: supported when the 95% CI of the overconfidence gap lies above 0 **and** the 95% CIs of ECE(stated) − ECE(prob) and ECE(stated) − ECE(agree) both lie above 0. Effect sizes with CIs are always reported. H1/H2/H3 are evaluated on the primary (complete-case, CoT) analysis; Direct results are secondary.

**Validity flags.** A model (per setting) is flagged, reported separately and excluded from the main comparison if any of these hold:

- accuracy < 40%;
- answer parse failure > 10% of CoT greedy outputs;
- median letter_mass < 0.5;
- truncation (finish_reason == "length") > 10% of CoT samples.

Parse failure and truncation are properties of the CoT generations and apply to both settings of that model.

## 9. Sanity checks (written to `sanity_report.md`)

[fail] = stops the run.

- [fail] dataset counts and schema; [fail] NaN or inf in any returned logprob.
- Parse-failure rates (greedy answer; greedy confidence before and after recovery; samples).
- letter_mass distribution, rate of any letter missing from top-k, CoT parsed-vs-argmax mismatch rate.
- Truncation rates; chosen vs gold letter distribution; per-signal histogram counts and fraction of tied values (and the 5 most common conf_stated values).
- Greedy determinism: greedy CoT re-run on 10 questions, number of identical outputs.
- Validity flags.

## 10. Pilot

20 train questions (section 2), same pipeline, used only to check formatting, parse rates, letter_mass, NaN/garbage output and throughput. The run aborts if the projected full runtime exceeds 11 hours. Pilot results are not part of the study.


In [ ]:
import glob, os, sys, time
MOCK = os.environ.get("MEDQA_MOCK") == "1"
INPUTS = ["/kaggle/input"] if os.path.isdir("/kaggle/input") else [os.environ.get("MEDQA_COMBINE_INPUT", "outputs")]
print("Python", sys.version.split()[0])
sys.path.insert(0, os.path.abspath("medqa_src"))
from medqa_conf import report
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
csvs = report._find_csvs(INPUTS)
zips = [] if csvs else sorted(glob.glob(os.path.join(INPUTS[0], "**", "share_bundle_*.zip"), recursive=True))
zips = [z for z in zips if not z.endswith("share_bundle_all.zip")]
print("found CSVs:", csvs, "\nfound bundles:", zips)
t0 = time.time()
C = report.combine(csvs or zips, os.path.join(WORK, "combined"), prereg_path="medqa_src/PREREGISTRATION.md")
print(open(os.path.join(WORK, "combined", "results_all.md"), encoding="utf-8").read())
print("models:", C["models"], "| bundle:", C["bundle"], f"| {time.time() - t0:.1f}s")
